# **Molecular States and Drug Sensitivities Associated with Gene Expression Changes in Recurrent Glioblastoma**

**Reference:** **Varn, F. S. et al. (2022). Glioma progression is shaped by genetic evolution and microenvironment interactions.** _Cell_, 185(12), **2184-2199.e16**. https://doi.org/10.1016/j.cell.2022.04.038
@articlevarnGliomaProgressionShaped2022



**Initial analysis**
1. Load GLASS data
2. Explore the dataset
3. Check raw gene expression data
4. Clean clinical data
5. Check patient–sample relationships — `glass_all_samples.xlsx`
6. Create primary–first recurrence pairs
7. Check gene expression data availability — `glass_primary_R1_pairs.xlsx`, `glass_primary_R1_pairs_all.xlsx`
8. Create the IDH-wildtype glioblastoma cohort — `glass_idhwt_gbm_primary_R1_pairs.xlsx`
9. Assess treatment group feasibility — `glass_treatment_variables.xlsx`
10. Summarise the initial analysis - `glass_patient_selection.png`, `treatment_overlap_venn.png`


**Gene expression processing and paired analysis**
1. Finalise the study cohort
2. Prepare the gene expression matrix - `glass_idhwt_gbm_primary_R1_TPM.tsv`
3. Perform initial quality control of gene expression data - `glass_expression_QC.xlsx`
4. Log-transform TPM data - `glass_idhwt_gbm_primary_R1_log2TPM.tvs`
5. Filter lowly expressed genes - `glass_gene_filter_summary.xlsx`, `glass_idhwt_gbm_primary_R1_log2TPM_filtered.tvs`
6. Assess global gene expression patterns across samples - `PCA_primary_recurrence.png`
7. Perform paired comparison of gene expression between primary and recurrent samples - `glass_primary_vs_R1_gene_statistics.xlsx`
8. Assess the consistency of expression changes
9. Generate an initial list of recurrence-associated genes -`glass_recurrence_UP_initial.xlsx`, `glass_recurrence_DOWN_initial.xlsx`
10. Generate initial result plots - `glass_recurrence_vulcano.png`, `glass_recurrence_heatmap.png` `glass_recurrence_paired_plots.png`

**Identifying biological processes associated with these changes**
1. Pathway analysis: Identifying biological processes that change during recurrence - `glass_GSEA_Hallmark.xlsx`, `glass_GSEA_Reactome.xlsx`
2. Summarising pathway analysis results
3. Assessing the stability of recurrence-associated genes using bootstrap analysis
4. Determining bootstrap selection frequency
5. Defining the recurrence signature
6. Creating the Recurrence Score
7. Initial assessment of the Recurrence Score in the full GLASS cohort
8. Out-of-fold validation using 5-fold cross-validation
9. Assessing the out-of-fold Recurrence Score
10. Sensitivity analysis in patients receiving standard therapy

**Can the transcriptional program associated with tumour recurrence be identified in glioblastoma cell lines?**
1. Solidifying Recurrence Score
2. Defining the final glioma cell-line cohort + quality check
3. Glioma cell-line master table
4. Merging GLASS signature and DepMap expression data
5. Calculating DepMap Recurrence Score
6. Verifying variance of DepMap Recurrence Score
7. Creating GDSC microarray Recurrence Score
8. Cross-platform validation of GDSC microarray and DepMap RNA-seq
9. Final checking of sensitivity data availability

In [ ]:
import pandas as pd
from graphviz import Digraph
from matplotlib_venn import venn3
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.path as mpath
import math
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests
from scipy.stats import t, skew, ttest_rel
import seaborn as sns
import gseapy as gp
import re

# **Initial analysis**

## 1. Load GLASS data
- **Source:** https://www.cbioportal.org/study/summary?id=difg_glass
    - https://datahub.assets.cbioportal.org/difg_glass.tar.gz
- **Raw Data:** `digf_glass.tar.tar`
- `difg_glass`
    - `README.md`
    - `LICENSE`
    - `data_clinical_patient.txt`
    - `data_clinical_sample.txt`
    - `data_cna.txt`
    - `data_cna_hg19.seg`
    - `data_mrna_seq_tpm.txt`
    - `data_mrna_seq_tpm_zscores_ref_all_samples.txt`
    - `data_mutations.txt`
    - `data_mutations_1.txt`
    - `data_mutations_2.txt`
    - `data_resource_definition.txt`
    - `data_resource_sample.txt`
    - `data_timeline_surgery.txt`
    - `data_timeline_treatment.txt`
    - `meta_clinical_patient.txt`
    - `meta_clinical_sample.txt`
    - `meta_cna.txt`
    - `meta_cna_hg19_seg.txt`
    - `meta_mrna_seq_tpm.txt`
    - `meta_mrna_seq_tpm_zscores_ref_all_samples.txt`
    - `meta_mutations.txt`
    - `meta_resource_definition.txt`
    - `meta_resource_sample.txt`
    - `meta_study.txt`
    - `meta_timeline_surgery.txt`
    - `meta_timeline_treatment.txt`
    - `case_lists`
        - `cases_all.txt`
        - `cases_cna.txt`
        - `cases_cnaseq.txt`
        - `cases_RNA_Seq_mRNA.txt`
        - `cases_sequenced.txt`

In [ ]:
data_clinical_patient = pd.read_csv(
    "../data/raw/difg_glass/data_clinical_patient.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

with open("../data/raw/difg_glass/data_clinical_patient.txt") as f:
    comments_p = [
        line[1:].rstrip("\n").split("\t")
        for line in f
        if line.startswith("#")
    ]

data_clinical_patient_comments = pd.DataFrame(comments_p[1:], columns=comments_p[0])

In [ ]:
data_clinical_sample = pd.read_csv(
    "../data/raw/difg_glass/data_clinical_sample.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

with open("../data/raw/difg_glass/data_clinical_sample.txt") as f:
    comments_s = [
        line[1:].rstrip("\n").split("\t")
        for line in f
        if line.startswith("#")
    ]

data_clinical_sample_comments = pd.DataFrame(comments_s[1:], columns=comments_s[0])

In [ ]:
data_cna = pd.read_csv(
    "../data/raw/difg_glass/data_cna.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

In [ ]:
data_cna_hg19 = pd.read_csv(
    "../data/raw/difg_glass/data_cna_hg19.seg",
    sep="\t",
    comment="#",
    low_memory=False
)

In [ ]:
data_mrna_seq_tpm = pd.read_csv(
    "../data/raw/difg_glass/data_mrna_seq_tpm.txt",
    sep="\t",
    comment="#",
    index_col=0,
    low_memory=False
)

In [ ]:
data_mrna_seq_tpm_zscores_ref_all_samples = pd.read_csv(
    "../data/raw/difg_glass/data_mrna_seq_tpm_zscores_ref_all_samples.txt",
    sep="\t",
    comment="#",
    index_col=0,
    low_memory=False
)

In [ ]:
data_mutations = pd.read_csv(
    "../data/raw/difg_glass/data_mutations.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

In [ ]:
data_mutations_1 = pd.read_csv(
    "../data/raw/difg_glass/data_mutations_1.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

In [ ]:
data_mutations_2 = pd.read_csv(
    "../data/raw/difg_glass/data_mutations_2.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

In [ ]:
data_resource_definition = pd.read_csv(
    "../data/raw/difg_glass/data_resource_definition.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

In [ ]:
data_resource_sample = pd.read_csv(
    "../data/raw/difg_glass/data_resource_sample.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

In [ ]:
data_timeline_surgery = pd.read_csv(
    "../data/raw/difg_glass/data_timeline_surgery.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

In [ ]:
data_timeline_treatment = pd.read_csv(
    "../data/raw/difg_glass/data_timeline_treatment.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

In [ ]:
meta_clinical_patient = pd.read_csv(
    "../data/raw/difg_glass/meta_clinical_patient.txt",
    sep="\t",
    low_memory=False
)

In [ ]:
meta_clinical_sample = pd.read_csv(
    "../data/raw/difg_glass/meta_clinical_sample.txt",
    sep="\t",
    low_memory=False
)

In [ ]:
meta_cna = pd.read_csv(
    "../data/raw/difg_glass/meta_cna.txt",
    sep="\t",
    low_memory=False
)

In [ ]:
meta_cna_hg19_seg = pd.read_csv(
    "../data/raw/difg_glass/meta_cna_hg19_seg.txt",
    sep="\t",
    low_memory=False
)

In [ ]:
meta_mrna_seq_tpm = pd.read_csv(
    "../data/raw/difg_glass/meta_mrna_seq_tpm.txt",
    sep="\t",
    low_memory=False
)

In [ ]:
meta_mrna_seq_tpm_zscores_ref_all_samples = pd.read_csv(
    "../data/raw/difg_glass/meta_mrna_seq_tpm_zscores_ref_all_samples.txt",
    sep="\t",
    low_memory=False
)

In [ ]:
meta_mutations = pd.read_csv(
    "../data/raw/difg_glass/meta_mutations.txt",
    sep="\t",
    low_memory=False
)

In [ ]:
meta_resource_definition = pd.read_csv(
    "../data/raw/difg_glass/meta_resource_definition.txt",
    sep="\t",
    low_memory=False
)

In [ ]:
meta_resource_sample = pd.read_csv(
    "../data/raw/difg_glass/meta_resource_sample.txt",
    sep="\t",
    low_memory=False
)

In [ ]:
meta_study = pd.read_csv(
    "../data/raw/difg_glass/meta_study.txt",
    sep="\t",
    low_memory=False
)

In [ ]:
meta_timeline_surgery = pd.read_csv(
    "../data/raw/difg_glass/meta_timeline_surgery.txt",
    sep="\t",
    low_memory=False
)

In [ ]:
meta_timeline_treatment = pd.read_csv(
    "../data/raw/difg_glass/meta_timeline_treatment.txt",
    sep="\t",
    low_memory=False
)

In [ ]:
cases_all = pd.read_csv(
    "../data/raw/difg_glass/case_lists/cases_all.txt",
    low_memory=False
)

In [ ]:
cases_cna = pd.read_csv(
    "../data/raw/difg_glass/case_lists/cases_cna.txt",
    low_memory=False
)

In [ ]:
cases_cnaseq = pd.read_csv(
    "../data/raw/difg_glass/case_lists/cases_cnaseq.txt",
    low_memory=False
)

In [ ]:
cases_RNA_Seq_mRNA = pd.read_csv(
    "../data/raw/difg_glass/case_lists/cases_RNA_Seq_mRNA.txt",
    low_memory=False
)

In [ ]:
cases_sequenced = pd.read_csv(
    "../data/raw/difg_glass/case_lists/cases_sequenced.txt",
    low_memory=False
)

## 2. Explore the dataset
- `data_*` are data
- `meta_*` are metadata
- `cases_*` number of cases for which the condition is true
- `README.md` contains the resources and pipelines used to compile this dataset

In [ ]:
meta_clinical_patient

In [ ]:
data_clinical_patient_comments

In [ ]:
cols = ["PATIENT_ID",
        "CASE_PROJECT",
        "SEX",
        "AGE",
        "OS_STATUS",
        "TISSUE_SOURCE"]

num_cols = data_clinical_patient.columns[~data_clinical_patient.columns.isin(cols)]

data_clinical_patient[cols] = data_clinical_patient[cols].astype("string")

for col in num_cols:
    data_clinical_patient[col] = pd.to_numeric(
        data_clinical_patient[col],
        errors="coerce"
    )

In [ ]:
data_clinical_patient.head()

- `data_clinical_patient.txt` - Clinical patient attributes
    - `PATIENT_ID`
        - Unique patient identifier
    - `CASE_PROJECT`
        - Indicates whether the data was collected for the purpose of GLASS or TCGA
    - `SEX`
        - Sex
    - `AGE`
        - Patient's age at first diagnosis
    - `OS_STATUS`
        - Overall patient survival status
    - `OS_MONTHS`
        - Overall survival in months since first diagnosis to last known follow-up
    - `TISSUE_SOURCE`
        - Tissue source site for each subject

In [ ]:
meta_clinical_sample

In [ ]:
data_clinical_sample_comments

In [ ]:
cases_all

In [ ]:
cols = ["PATIENT_ID",
        "SAMPLE_ID",
        "SAMPLE_TYPE",
        "ONCOTREE_CODE",
        "HISTOLOGY",
        "TUMOR_GRADE",
        "IDH_STATUS",
        "IDH_CODEL_STATUS",
        "TUMOR_CLASSIFICATION",
        "CODEL_STATUS",
        "MGMT_METHYLATION",
        "SURGERY_TYPE",
        "SURGERY_INDICATION",
        "SURGERY_EXTENT_OF_RESECTION",
        "SURGERY_LATERALITY",
        "SURGERY_LOCATION",
        "TREATMENT_TMZ",
        "TREATMENT_TMZ_CYCLES_6",
        "TREATMENT_CONCURRENT_TMZ",
        "TREATMENT_RADIOTHERAPY",
        "ALKYLATING_AGENT_TX",
        "MGMT_METHYLATION_METHOD",
        "DNA_ALIQUOT_BARCODE",
        "RNA_ALIQUOT_BARCODE",
        "ALIQUOT_ANALYSIS_TYPE",
        "CANCER_TYPE",
        "CANCER_TYPE_DETAILED"]

num_cols = data_clinical_sample.columns[~data_clinical_sample.columns.isin(cols)]

data_clinical_sample[cols] = data_clinical_sample[cols].astype("string")

for col in num_cols:
    data_clinical_sample[col] = pd.to_numeric(
        data_clinical_sample[col],
        errors="coerce"
    )

In [ ]:
data_clinical_sample.groupby(["PATIENT_ID", "SAMPLE_TYPE"])["SAMPLE_ID"].nunique().value_counts()

- All samples are unique; each patient has at most one sample per sample type.

In [ ]:
data_clinical_sample.head()

- All unique samples in study: 693
- `data_clinical_sample.txt` - Clinical sample attributes
    - `PATIENT_ID`
        - Unique patient identifier
    - `SAMPLE_ID`
        - Unique sample identifier
    - `SAMPLE_TYPE`
        - Sample Type
    - `ONCOTREE_CODE`
        - Oncotree code
    - `HISTOLOGY`
        - Histology classification
    -  `TUMOR_GRADE`
        - Pathological tumor grade
    -  `IDH_STATUS`
        - IDH Status
    -  `IDH_CODEL_STATUS`
        - IDH and Codeletion Status
        - Combination of the idh_status and codel_status variables
    - `TUMOR_CLASSIFICATION`
        - Tumour classification according to WHO
    -  `CODEL_STATUS`
        - Codeletion status of chromosomal arms 1p and 19q
    -  `MGMT_METHYLATION`
        - Clinical assessment of MGMT methylation in samples
    - `SURGERY_TYPE`
        - Surgery type
    -  `SURGERY_INDICATION`
        - Surgery indication
    - `SURGERY_EXTENT_OF_RESECTION`
        - Indicated extent of resection needed for sample collection
    - `SURGERY_LATERALITY`
        - Hemisphere from which the tumour was resected
    - `SURGERY_LOCATION`
        - Brain region from which tumour was resected
    - `TREATMENT_TMZ`
        - Indicates whether a subject received temozolomide (tmz) for each surgical sample
    - `TREATMENT_TMZ_CYCLES`
        - Number of temozolomide treatment cycles
    -  `TREATMENT_TMZ_CYCLES_6`
        - Indicates whether a subject received at least six cycles of temozolomide
    -  `TREATMENT_CONCURRENT_TMZ`
        - Indicate whether temozolomide was administered concurrently with radiotherapy
    -  `TREATMENT_RADIOTHERAPY`
        - Indicates whether a subject received radiotherapy
    -  `TREATMENT_RADIATION_DOSE_GY`
        - Radiation Dose (Gy)
    - `ALKYLATING_AGENT_TX`
        - Treatment with alkylating agents
    - `MGMT_METHYLATION_METHOD`
        - MGMT methylation method
    - `DNA_ALIQUOT_BARCODE`
        - A variable that stitches subject, sample type, portion, sequencing type, and a unique identifier together
    - `RNA_ALIQUOT_BARCODE`
        - A variable that stitches subject, sample type, portion, sequencing type, and a unique identifier together
    - `ALIQUOT_ANALYSIS_TYPE`
        - Variable representing either whole genome sequencing (WGS) or whole exome sequencing (WXS)
    - `STROMAL_SCORE`
        - An ssGSEA-based enrichment score calculated from the ESTIMATE stromal signature. Represents stromal content in a given tumour.
    - `IMMUNE_SCORE`
        - An ssGSEA-based enrichment score calculated from the ESTIMATE immune signature. Represents general immune content in a given tumour.
    - `ESTIMATE_SCORE`
        - The ESTIMATE score that is calculated using both the stromal and immune score that represents non-tumour content in a tumour.
    - `PURITY`
        - The estimated tumour purity value calculated from the estimate_score.
    - `CANCER_TYPE`
        - Cancer Type
    - `CANCER_TYPE_DETAILED`
        - Cancer Type Detailed

In [ ]:
meta_cna

In [ ]:
cases_cna

In [ ]:
data_cna.iloc[:5, :5]

- Samples with CNA data: 629 samples (`cases_cna.txt`)
- `data_cna.txt`
    - Copy number alterations based on GATK segmentation output.

In [ ]:
meta_cna_hg19_seg

In [ ]:
cases_cnaseq

In [ ]:
data_cna_hg19.head()

- Samples with mutation and CNA data: 629 samples (`cases_cnaseq.txt`)
- `data_cna_hg19.seg`
    - Segment data values

In [ ]:
meta_mrna_seq_tpm

In [ ]:
cases_RNA_Seq_mRNA

In [ ]:
data_mrna_seq_tpm.iloc[:5, :5]

- Samples with mRNA expression data: 355 samples (`cases_RNA_Seq_mRNA.txt`)
- `data_mrna_seq_tpm.txt`
    - Expression levels from 168 diffuse glioma cases (RNA-Seq TPM). Values as calculated by Kallisto.

In [ ]:
meta_mrna_seq_tpm_zscores_ref_all_samples

In [ ]:
data_mrna_seq_tpm_zscores_ref_all_samples.iloc[:5, :5]

- `data_mrna_seq_tpm_zscores_ref_all_samples.txt`
    - Log-transformed mRNA z-scores compared to the expression distribution of all samples (RNA Seq TPM).

In [ ]:
meta_mutations

In [ ]:
cases_sequenced

In [ ]:
data_mutations.head()

In [ ]:
data_mutations_1.head()

In [ ]:
data_mutations_2.head()

- Samples with mutation data: 629 samples (`cases_sequenced.txt`)
- `data_mutations.txt`, `data_mutations_1.txt`, `data_mutations_2.txt`
    -  Whole genome or whole exome sequencing of diffuse glioma tumour/normal pairs from 299 patients.

In [ ]:
meta_resource_sample

In [ ]:
data_resource_sample.head()

- `data_resource_sample.txt`
    - Data Resources

In [ ]:
meta_timeline_surgery

In [ ]:
cols = ["PATIENT_ID",
        "EVENT_TYPE",
        "SURGERY_TYPE",
        "SURGERY_INDICATION",
        "SURGERY_EXTENT_OF_RESECTION",
        "SURGERY_LATERALITY",
        "SURGERY_LOCATION",
        "SAMPLE_ID"]

num_cols = data_timeline_surgery.columns[~data_timeline_surgery.columns.isin(cols)]

data_timeline_surgery[cols] = data_timeline_surgery[cols].astype("string")

for col in num_cols:
   data_timeline_surgery[col] = pd.to_numeric(
        data_timeline_surgery[col],
        errors="coerce"
    )

In [ ]:
data_timeline_surgery.head()

- `data_timeline_surgery.txt`
    - Surgery timeline and information of patients

In [ ]:
meta_timeline_treatment

In [ ]:
cols = [
    "PATIENT_ID",
    "EVENT_TYPE",
    "TREATMENT_TYPE",
    "TREATMENT_SUBTYPE",
    "AGENT",
    "TREATMENT_CYCLES_6",
    "TREATMENT_CYCLES_NOTES",
    "RADIATION_OTHER"
]

num_cols = data_timeline_treatment.columns[~data_timeline_treatment.columns.isin(cols)]

data_timeline_treatment[cols] = data_timeline_treatment[cols].astype("string")

for col in num_cols:
   data_timeline_treatment[col] = pd.to_numeric(
        data_timeline_treatment[col],
        errors="coerce"
    )

In [ ]:
data_timeline_treatment.head()

- `data_timeline_treatment.txt`
    - Treatment timeline and information of patients

## 3. Check raw gene expression data
- `data_mrna_seq_tpm.txt`

In [ ]:
for col in data_mrna_seq_tpm.columns:
    data_mrna_seq_tpm[col] = pd.to_numeric(
        data_mrna_seq_tpm[col],
        errors="coerce"
    )

In [ ]:
data_mrna_seq_tpm.head()

In [ ]:
glass_expr = data_mrna_seq_tpm.copy()

In [ ]:
glass_expr.head()

## 4. Clean clinical data
- `data_clinical_patient.txt`
- `data_clinical_sample.txt`
- Was done when loading dataset

## 5. Check patient–sample relationships - `glass_all_samples.xlsx`
- `data_clinical_sample`: `PATIENT_ID`, `SAMPLE_ID`, `SAMPLE_TYPE`,
  `TUMOR_GRADE`, `TUMOR_CLASSIFICATION`, `IDH_CODEL_STATUS`,
  `CANCER_TYPE_DETAILED`
- `data_timeline_surgery`:
    - use `START_DATE` to derive:
        - `SURGERY_NUMBER`
        - `COLLECTION_TIME`

In [ ]:
data_clinical_sample.head()

In [ ]:
data_timeline_surgery.head()

In [ ]:
glass_all_samples_raw = pd.merge(
    data_clinical_sample,
    data_timeline_surgery,
    on=["SAMPLE_ID", "PATIENT_ID", "SURGERY_TYPE", "SURGERY_INDICATION", "SURGERY_EXTENT_OF_RESECTION", "SURGERY_LATERALITY", "SURGERY_LOCATION"],
    how="outer"
)

In [ ]:
glass_all_samples_raw.head()

In [ ]:
glass_all_samples = glass_all_samples_raw[[
    "PATIENT_ID",
    "SAMPLE_ID",
    "SAMPLE_TYPE",
    "TUMOR_GRADE",
    "TUMOR_CLASSIFICATION",
    "IDH_CODEL_STATUS",
    "CANCER_TYPE_DETAILED",
    "START_DATE"
]].copy()

In [ ]:
glass_all_samples["SURGERY_NUMBER"] = (
    glass_all_samples.groupby("PATIENT_ID")["START_DATE"]
      .rank(method="first", ascending=True)
      .astype("Int64")
)

In [ ]:
glass_all_samples["COLLECTION_TIME"] = glass_all_samples["START_DATE"].copy()
glass_all_samples = glass_all_samples.drop(columns=["START_DATE"])

In [ ]:
glass_all_samples[["DIAGNOSIS", "TUMOR_IDH_STATUS"]] = (
    glass_all_samples["TUMOR_CLASSIFICATION"]
    .str.split(",", n=1, expand=True)
)

glass_all_samples["DIAGNOSIS"] = glass_all_samples["DIAGNOSIS"].str.strip()
glass_all_samples["TUMOR_IDH_STATUS"] = glass_all_samples["TUMOR_IDH_STATUS"].str.strip()

In [ ]:
# TUMOR_IDH_STATUS was changed to be data_clinical_sample[IDK_CODEL_STATUS]
glass_all_samples["TUMOR_IDH_STATUS"] = glass_all_samples["IDH_CODEL_STATUS"].copy()

In [ ]:
glass_all_samples = glass_all_samples[[
    "PATIENT_ID",
    "SAMPLE_ID",
    "CANCER_TYPE_DETAILED",
    "SAMPLE_TYPE",
    "SURGERY_NUMBER",
    "COLLECTION_TIME",
    "DIAGNOSIS",
    "TUMOR_GRADE",
    "TUMOR_IDH_STATUS"
]]

In [ ]:
glass_all_samples.head()

In [ ]:
glass_all_samples.to_excel("../data/processed/glass_all_samples.xlsx", index=False)

- Final `glass_all_samples.xlsx` exported to `/data/processed` directory

## 6. Create primary-first recurrent pairs - `glass_primary_R1_pairs_all`
- One row per patient
- Must contain both:
    - `SAMPLE_TYPE = Tumor Primary`
    - `SAMPLE_TYPE = First Recurrence`
- Other recurrence types (R2, R3, R4, etc.) do not affect eligibility
- Retain:
    - `PATIENT_ID`
    - `PRIMARY_SAMPLE_ID`
    - `FIRST_RECURRENCE_SAMPLE_ID`

In [ ]:
glass_primary_R1_pairs_all = glass_all_samples[[
    "PATIENT_ID",
    "SAMPLE_ID",
    "SAMPLE_TYPE"
]].copy()

In [ ]:
glass_primary_R1_pairs_all = glass_primary_R1_pairs_all.loc[
    glass_primary_R1_pairs_all["SAMPLE_TYPE"].isin(
        ["Tumor Primary", "First Recurrence"]),
        ["PATIENT_ID", "SAMPLE_ID", "SAMPLE_TYPE"]].copy()

In [ ]:
glass_primary_R1_pairs_all["PRIMARY_SAMPLE_ID"] = glass_primary_R1_pairs_all["SAMPLE_ID"].where(
    glass_primary_R1_pairs_all["SAMPLE_TYPE"] == "Tumor Primary"
)

glass_primary_R1_pairs_all["FIRST_RECURRENCE_SAMPLE_ID"] = glass_primary_R1_pairs_all["SAMPLE_ID"].where(
    glass_primary_R1_pairs_all["SAMPLE_TYPE"] == "First Recurrence"
)

In [ ]:
glass_primary_R1_pairs_all.drop(
columns=["SAMPLE_TYPE", "SAMPLE_ID"],
inplace=True
)

In [ ]:
glass_primary_R1_pairs_all = (
    glass_primary_R1_pairs_all
    .groupby("PATIENT_ID", as_index=False)
    .agg({
        "PRIMARY_SAMPLE_ID": "first",
        "FIRST_RECURRENCE_SAMPLE_ID": "first"
    })
    .dropna(subset=["PRIMARY_SAMPLE_ID", "FIRST_RECURRENCE_SAMPLE_ID"])
)

## 7. Check gene expression data availability - `glass_primary_R1_pairs_all.xlsx`, `glass_primary_R1_pairs.xlsx`
- Cross-reference `PRIMARY_SAMPLE_ID` and `FIRST_RECURRENCE_SAMPLE_ID`
  from `glass_primary_R1_pairs_all` with `glass_expr.columns`
- `primary_has_expression`, `recurrent_has_expression`
- `glass_primary_R1_pairs_all.xlsx`
    - All patients with both primary and first recurrent samples
    - Includes expression availability flags
- `glass_primary_R1_pairs.xlsx`
    - Patients for whom both primary and first recurrent samples have expression data

In [ ]:
glass_expr.head()

In [ ]:
glass_primary_R1_pairs_all["primary_has_expression"] = (
    glass_primary_R1_pairs_all["PRIMARY_SAMPLE_ID"]
    .isin(glass_expr.columns)
)

glass_primary_R1_pairs_all["recurrent_has_expression"] = (
    glass_primary_R1_pairs_all["FIRST_RECURRENCE_SAMPLE_ID"]
    .isin(glass_expr.columns)
)

In [ ]:
glass_primary_R1_pairs_all["primary_and_recurrent_has_expression"] = (
    glass_primary_R1_pairs_all["primary_has_expression"]
    & glass_primary_R1_pairs_all["recurrent_has_expression"]
)

glass_primary_R1_pairs = glass_primary_R1_pairs_all.loc[
    glass_primary_R1_pairs_all["primary_and_recurrent_has_expression"]
].copy()

glass_primary_R1_pairs = glass_primary_R1_pairs[["PATIENT_ID", "PRIMARY_SAMPLE_ID", "FIRST_RECURRENCE_SAMPLE_ID"]].copy()

In [ ]:
glass_primary_R1_pairs_all.to_excel(
    "../data/processed/glass_primary_R1_pairs_all.xlsx",
    index=False
)

- Final `glass_primary_R1_pairs_all.xlsx` exported to `/data/processed` directory

In [ ]:
glass_primary_R1_pairs.to_excel(
    "../data/processed/glass_primary_R1_pairs.xlsx",
    index=False
)

- Final `glass_primary_R1_pairs.xlsx` exported to `/data/processed` directory


## 8. Create the IDH-wildtype glioblastoma cohort - `glass_idhwt_gbm_primary_R1_pairs.xlsx`
- Primary dataframe for analysis

In [ ]:
glass_all_samples.head()

In [ ]:
glass_primary_R1_pairs_diag_idh = glass_all_samples.copy()

glass_primary_R1_pairs_diag_idh["PRIMARY_DIAGNOSIS"] = glass_all_samples["DIAGNOSIS"].where(
    glass_all_samples["SAMPLE_TYPE"] == "Tumor Primary"
)

glass_primary_R1_pairs_diag_idh["FIRST_RECURRENCE_DIAGNOSIS"] = glass_all_samples["DIAGNOSIS"].where(
    glass_all_samples["SAMPLE_TYPE"] == "First Recurrence"
)

glass_primary_R1_pairs_diag_idh["PRIMARY_IDH"] = glass_all_samples["TUMOR_IDH_STATUS"].where(
    glass_all_samples["SAMPLE_TYPE"] == "Tumor Primary"
)

glass_primary_R1_pairs_diag_idh["FIRST_RECURRENCE_IDH"] = glass_all_samples["TUMOR_IDH_STATUS"].where(
    glass_all_samples["SAMPLE_TYPE"] == "First Recurrence"
)

In [ ]:
glass_primary_R1_pairs_diag_idh.drop(
columns=["SAMPLE_TYPE", "SAMPLE_ID", "CANCER_TYPE_DETAILED", "SURGERY_NUMBER", "COLLECTION_TIME", "DIAGNOSIS", "TUMOR_GRADE", "TUMOR_IDH_STATUS"],
inplace=True
)

In [ ]:
glass_primary_R1_pairs_diag_idh = (
    glass_primary_R1_pairs_diag_idh
    .groupby("PATIENT_ID", as_index=False)
    .agg({
        "PRIMARY_DIAGNOSIS": "first",
        "FIRST_RECURRENCE_DIAGNOSIS": "first",
        "PRIMARY_IDH": "first",
        "FIRST_RECURRENCE_IDH": "first",
    })
)

In [ ]:
glass_idh_diag_primary_R1_pairs = glass_primary_R1_pairs.merge(
    glass_primary_R1_pairs_diag_idh,
    on="PATIENT_ID",
    how="left"
)

In [ ]:
glass_idh_diag_primary_R1_pairs.head()

In [ ]:
glass_gbm_primary_R1_pairs = glass_idh_diag_primary_R1_pairs.loc[
    (glass_idh_diag_primary_R1_pairs["PRIMARY_DIAGNOSIS"] == "Glioblastoma") &
    (glass_idh_diag_primary_R1_pairs["FIRST_RECURRENCE_DIAGNOSIS"] == "Glioblastoma"),
    ["PATIENT_ID", "PRIMARY_SAMPLE_ID", "FIRST_RECURRENCE_SAMPLE_ID", "PRIMARY_IDH", "FIRST_RECURRENCE_IDH"]
].copy()

glass_idhwt_gbm_primary_R1_pairs = glass_gbm_primary_R1_pairs.loc[
    (glass_gbm_primary_R1_pairs["PRIMARY_IDH"] == "IDHwt") &
    (glass_gbm_primary_R1_pairs["FIRST_RECURRENCE_IDH"] == "IDHwt"),
    ["PATIENT_ID", "PRIMARY_SAMPLE_ID", "FIRST_RECURRENCE_SAMPLE_ID"]
].copy()

In [ ]:
glass_idhwt_gbm_primary_R1_pairs.head()

In [ ]:
glass_idhwt_gbm_primary_R1_pairs.to_excel(
    "../data/processed/glass_idhwt_gbm_primary_R1_pairs.xlsx", index=False)

- Final `glass_idhwt_gbm_primary_R1_pairs.xlsx` exported to `/data/processed` directory.

## 9. Assess treatment group feasibility - `glass_treatment_variables.xlsx`
- Will be used later to create treatment groups
- Variables of interest: temozolomide, radiotherapy, concurrent TMZ + radiotherapy,  other systemic treatments
- Can treatment exposure between the primary and first recurrent samples be determined from the available clinical and treatment timeline data?

In [ ]:
treatment_pairs = (glass_idhwt_gbm_primary_R1_pairs.copy())
assert treatment_pairs["PATIENT_ID"].is_unique

In [ ]:
sample_treatment = (
    glass_all_samples
    .merge(
        data_clinical_sample[
            [
                "PATIENT_ID",
                "SAMPLE_ID",
                "TREATMENT_TMZ",
                "TREATMENT_RADIOTHERAPY",
                "TREATMENT_CONCURRENT_TMZ",
            ]
        ],
        on=["PATIENT_ID", "SAMPLE_ID"],
        how="left",
    )
)

sample_treatment = sample_treatment[
    [
        "PATIENT_ID",
        "SAMPLE_ID",
        "SAMPLE_TYPE",
        "COLLECTION_TIME",
        "TREATMENT_TMZ",
        "TREATMENT_RADIOTHERAPY",
        "TREATMENT_CONCURRENT_TMZ",
    ]
].copy()

sample_treatment = sample_treatment.loc[
    sample_treatment["PATIENT_ID"].isin(
        treatment_pairs["PATIENT_ID"]
    )
].copy()

In [ ]:
sample_times = (
    sample_treatment[
        [
            "PATIENT_ID",
            "SAMPLE_ID",
            "SAMPLE_TYPE",
            "COLLECTION_TIME",
        ]
    ]
    .drop_duplicates()
)

primary_times = (
    sample_times.loc[
        sample_times["SAMPLE_TYPE"] == "Tumor Primary",
        [
            "PATIENT_ID",
            "SAMPLE_ID",
            "COLLECTION_TIME",
        ],
    ]
    .rename(
        columns={
            "SAMPLE_ID": "PRIMARY_SAMPLE_ID",
            "COLLECTION_TIME": "PRIMARY_COLLECTION_TIME",
        }
    )
)

r1_times = (
    sample_times.loc[
        sample_times["SAMPLE_TYPE"] == "First Recurrence",
        [
            "PATIENT_ID",
            "SAMPLE_ID",
            "COLLECTION_TIME",
        ],
    ]
    .rename(
        columns={
            "SAMPLE_ID": "FIRST_RECURRENCE_SAMPLE_ID",
            "COLLECTION_TIME": "FIRST_RECURRENCE_COLLECTION_TIME",
        }
    )
)

treatment_pairs = (
    treatment_pairs
    .merge(
        primary_times,
        on=["PATIENT_ID", "PRIMARY_SAMPLE_ID"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        r1_times,
        on=["PATIENT_ID", "FIRST_RECURRENCE_SAMPLE_ID"],
        how="left",
        validate="one_to_one",
    )
)

treatment_pairs["PRIMARY_COLLECTION_TIME"] = pd.to_numeric(
    treatment_pairs["PRIMARY_COLLECTION_TIME"],
    errors="coerce",
)

treatment_pairs["FIRST_RECURRENCE_COLLECTION_TIME"] = pd.to_numeric(
    treatment_pairs["FIRST_RECURRENCE_COLLECTION_TIME"],
    errors="coerce",
)

treatment_pairs["INTERVAL_ASSESSABLE"] = (
    treatment_pairs["PRIMARY_COLLECTION_TIME"].notna()
    & treatment_pairs["FIRST_RECURRENCE_COLLECTION_TIME"].notna()
    & (
        treatment_pairs["FIRST_RECURRENCE_COLLECTION_TIME"]
        > treatment_pairs["PRIMARY_COLLECTION_TIME"]
    )
)

In [ ]:
sample_type_map = {
    "Tumor Primary": "PRIMARY",
    "First Recurrence": "FIRST_RECURRENCE",
}

treatment_map = {
    "TREATMENT_TMZ": "TMZ",
    "TREATMENT_RADIOTHERAPY": "RADIOTHERAPY",
    "TREATMENT_CONCURRENT_TMZ": "CONCURRENT_TMZ",
}

for sample_type, prefix in sample_type_map.items():

    for source_col, output_name in treatment_map.items():

        status = (
            sample_treatment.loc[
                sample_treatment["SAMPLE_TYPE"] == sample_type,
                ["PATIENT_ID", source_col],
            ]
            .drop_duplicates("PATIENT_ID")
            .set_index("PATIENT_ID")[source_col]
            .map({
                "Yes": True,
                "No": False,
            })
            .astype("boolean")
        )

        treatment_pairs[f"{prefix}_{output_name}"] = (
            treatment_pairs["PATIENT_ID"]
            .map(status)
            .astype("boolean")
        )

In [ ]:
timeline = data_timeline_treatment.copy()

timeline["START_DATE"] = pd.to_numeric(
    timeline["START_DATE"],
    errors="coerce",
)

treatment_between = (
    timeline
    .merge(
        treatment_pairs[
            [
                "PATIENT_ID",
                "PRIMARY_COLLECTION_TIME",
                "FIRST_RECURRENCE_COLLECTION_TIME",
                "INTERVAL_ASSESSABLE",
            ]
        ],
        on="PATIENT_ID",
        how="inner",
    )
)

treatment_between = treatment_between.loc[
    treatment_between["INTERVAL_ASSESSABLE"]
    & treatment_between["START_DATE"].notna()
    & (
        treatment_between["START_DATE"]
        >= treatment_between["PRIMARY_COLLECTION_TIME"]
    )
    & (
        treatment_between["START_DATE"]
        < treatment_between["FIRST_RECURRENCE_COLLECTION_TIME"]
    )
].copy()

In [ ]:
agent_clean = (
    treatment_between["AGENT"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

treatment_between["IS_TMZ"] = agent_clean.isin(
    [
        "temozolomide",
        "temodar",
        "tmz",
    ]
)

treatment_between["IS_RADIOTHERAPY"] = (
    treatment_between["TREATMENT_TYPE"]
    == "Radiation Therapy"
)

In [ ]:
tmz_summary = (
    treatment_between
    .groupby("PATIENT_ID")["IS_TMZ"]
    .any()
    .rename("RECEIVED_TMZ_BETWEEN")
    .reset_index()
)

In [ ]:
rt_summary = (
    treatment_between
    .groupby("PATIENT_ID")["IS_RADIOTHERAPY"]
    .any()
    .rename("RECEIVED_RADIOTHERAPY_BETWEEN")
    .reset_index()
)

In [ ]:
tmz_dates = (
    treatment_between.loc[
        treatment_between["IS_TMZ"],
        ["PATIENT_ID", "START_DATE"],
    ]
    .drop_duplicates()
)

rt_dates = (
    treatment_between.loc[
        treatment_between["IS_RADIOTHERAPY"],
        ["PATIENT_ID", "START_DATE"],
    ]
    .drop_duplicates()
)

concurrent_patients = (
    tmz_dates
    .merge(
        rt_dates,
        on=["PATIENT_ID", "START_DATE"],
        how="inner",
    )["PATIENT_ID"]
    .drop_duplicates()
)

In [ ]:
other_systemic = treatment_between.loc[
    (treatment_between["TREATMENT_TYPE"] == "Chemotherapy")
    & treatment_between["AGENT"].notna()
    & ~treatment_between["IS_TMZ"]
].copy()

other_systemic_summary = (
    other_systemic
    .groupby("PATIENT_ID")["AGENT"]
    .agg(
        lambda x: ", ".join(
            pd.unique(x.dropna().astype(str))
        )
    )
    .rename("OTHER_SYSTEMIC_AGENTS")
    .reset_index()
)

other_systemic_summary["OTHER_SYSTEMIC_TREATMENTS"] = True

In [ ]:
treatment_variables = (
    treatment_pairs
    .merge(
        tmz_summary,
        on="PATIENT_ID",
        how="left",
        validate="one_to_one",
    )
    .merge(
        rt_summary,
        on="PATIENT_ID",
        how="left",
        validate="one_to_one",
    )
    .merge(
        other_systemic_summary,
        on="PATIENT_ID",
        how="left",
        validate="one_to_one",
    )
)

In [ ]:
boolean_columns = [
    "PRIMARY_TMZ",
    "FIRST_RECURRENCE_TMZ",
    "RECEIVED_TMZ_BETWEEN",
    "PRIMARY_RADIOTHERAPY",
    "FIRST_RECURRENCE_RADIOTHERAPY",
    "RECEIVED_RADIOTHERAPY_BETWEEN",
    "PRIMARY_CONCURRENT_TMZ",
    "FIRST_RECURRENCE_CONCURRENT_TMZ",
]

for col in boolean_columns:
    treatment_variables[col] = (
        treatment_variables[col]
        .astype("boolean")
    )

treatment_variables["RECEIVED_TMZ"] = (
    treatment_variables["PRIMARY_TMZ"].fillna(False)
    | treatment_variables["FIRST_RECURRENCE_TMZ"].fillna(False)
    | treatment_variables["RECEIVED_TMZ_BETWEEN"].fillna(False)
)

treatment_variables["RECEIVED_RADIOTHERAPY"] = (
    treatment_variables["PRIMARY_RADIOTHERAPY"].fillna(False)
    | treatment_variables["FIRST_RECURRENCE_RADIOTHERAPY"].fillna(False)
    | treatment_variables["RECEIVED_RADIOTHERAPY_BETWEEN"].fillna(False)
)

treatment_variables["RECEIVED_CONCURRENT_TMZ"] = (
    treatment_variables["PRIMARY_CONCURRENT_TMZ"].fillna(False)
    | treatment_variables["FIRST_RECURRENCE_CONCURRENT_TMZ"].fillna(False)
    | treatment_variables["PATIENT_ID"].isin(concurrent_patients)
)

treatment_variables["OTHER_SYSTEMIC_TREATMENTS"] = (
    treatment_variables["OTHER_SYSTEMIC_TREATMENTS"]
    .astype("boolean")
    .fillna(False)
)

In [ ]:
final_columns = [
    "PATIENT_ID",
    "PRIMARY_SAMPLE_ID",
    "FIRST_RECURRENCE_SAMPLE_ID",
    "RECEIVED_TMZ",
    "RECEIVED_RADIOTHERAPY",
    "RECEIVED_CONCURRENT_TMZ",
    "OTHER_SYSTEMIC_TREATMENTS",
    "OTHER_SYSTEMIC_AGENTS",
    "PRIMARY_COLLECTION_TIME",
    "FIRST_RECURRENCE_COLLECTION_TIME",
    "PRIMARY_TMZ",
    "PRIMARY_RADIOTHERAPY",
    "PRIMARY_CONCURRENT_TMZ",
    "FIRST_RECURRENCE_TMZ",
    "FIRST_RECURRENCE_RADIOTHERAPY",
    "FIRST_RECURRENCE_CONCURRENT_TMZ",
]

treatment_variables = treatment_variables[
    final_columns
].reset_index(drop=True)

In [ ]:
treatment_variables.head()

In [ ]:
glass_treatment_variables = treatment_variables.copy()

glass_treatment_variables_complex = glass_treatment_variables [[
    "PATIENT_ID",
    "PRIMARY_SAMPLE_ID",
    "FIRST_RECURRENCE_SAMPLE_ID",
    "RECEIVED_TMZ",
    "RECEIVED_RADIOTHERAPY",
    "RECEIVED_CONCURRENT_TMZ",
    "OTHER_SYSTEMIC_TREATMENTS",
    "OTHER_SYSTEMIC_AGENTS",
    "PRIMARY_COLLECTION_TIME",
    "FIRST_RECURRENCE_COLLECTION_TIME",
    "PRIMARY_TMZ",
    "PRIMARY_RADIOTHERAPY",
    "PRIMARY_CONCURRENT_TMZ",
    "FIRST_RECURRENCE_TMZ",
    "FIRST_RECURRENCE_RADIOTHERAPY",
    "FIRST_RECURRENCE_CONCURRENT_TMZ"
]]

In [ ]:
glass_treatment_variables_simple = glass_treatment_variables [[
    "PATIENT_ID",
    "PRIMARY_SAMPLE_ID",
    "FIRST_RECURRENCE_SAMPLE_ID",
    "RECEIVED_TMZ",
    "RECEIVED_RADIOTHERAPY",
    "RECEIVED_CONCURRENT_TMZ",
    "OTHER_SYSTEMIC_TREATMENTS",
    "OTHER_SYSTEMIC_AGENTS"
]]

In [ ]:
with pd.ExcelWriter("../data/processed/glass_treatment_variables.xlsx") as writer:
    glass_treatment_variables_simple.to_excel(
        writer,
        sheet_name="Simple treatment variables",
        index=False
    )

    glass_treatment_variables_complex.to_excel(
        writer,
        sheet_name="Complex Treatment variables",
        index=False
    )

## 10. Summarise the initial analysis - `glass_patient_selection.png`, `treatment_overlap_venn.png`

In [ ]:
n_all = set(data_clinical_sample["PATIENT_ID"])

n_expr = set(glass_expr.columns)

sample_counts = (
    data_clinical_sample
    .groupby("PATIENT_ID")["SAMPLE_ID"]
    .nunique()
)

n_two_samples = set(sample_counts[sample_counts >= 2].index)

n_tp_r1 = set(glass_primary_R1_pairs_all["PATIENT_ID"])

n_tp_r1_expr = set(glass_primary_R1_pairs['PATIENT_ID'])

n_gbm = set(glass_gbm_primary_R1_pairs["PATIENT_ID"])

n_idhwt_gbm = set(glass_idhwt_gbm_primary_R1_pairs["PATIENT_ID"])

tmz = set(
    glass_treatment_variables.loc[
        glass_treatment_variables["RECEIVED_TMZ"],
        "PATIENT_ID"
    ]
)

radiotherapy = set(
    glass_treatment_variables.loc[
        glass_treatment_variables["RECEIVED_RADIOTHERAPY"],
        "PATIENT_ID"
    ]
)

other_systemic = set(
    glass_treatment_variables.loc[
        glass_treatment_variables["OTHER_SYSTEMIC_TREATMENTS"],
        "PATIENT_ID"
    ]
)

concurrent_tmz = set(
    glass_treatment_variables.loc[
        glass_treatment_variables["RECEIVED_CONCURRENT_TMZ"],
        "PATIENT_ID"
    ]
)

In [ ]:
print("Total number of patients in the GLASS study:", len(n_all))

print("Total number of samples in the GLASS study:", data_clinical_sample["SAMPLE_ID"].nunique())

print("Total number of samples with available expression data:", len(n_expr))

print("Number of patients with at least two samples:", len(n_two_samples))

print("Number of patient with tumor primary and first recurrent samples:", len(n_tp_r1))

print("Number of patient with tumor primary and first recurrent samples, with available expression data:", len(n_tp_r1_expr))

print("Number of patient with tumor primary and first recurrent samples, with available expression data and glioblastoma diagnosis:", len(n_gbm))

print("Number of patient with tumor primary and first recurrent samples, with available expression data and IDH-wildtype glioblastoma:", len(n_idhwt_gbm))

print("Out of these patients, has received TMZ treatment:", len(tmz))

print("Out of these patients, has received radiotherapy:", len(radiotherapy))

print("Out of these patients, has received concurrent TMZ treatment and radiotherapy:", len(concurrent_tmz))

print("Out of these patients, has received other systemic treatments:", len(other_systemic))

In [ ]:
flow = Digraph()

flow.attr(
    rankdir="TB",
    nodesep="0.5",
    ranksep="0.7"
)

flow.attr(
    "node",
    shape="box",
    style="rounded",
    fontname="Helvetica Neue",
    fontsize="11",
    margin="0.15"
)

flow.attr(
    "edge",
    fontname="Helvetica Neue",
    fontsize="9",
    arrowsize="0.7"
)

flow.node("A", f"All GLASS patients\nn = {len(n_all)}")
flow.node("B", f"Patients with\nTumor Primary + First Recurrence\nn = {len(n_tp_r1)}")
flow.node("C", f"Patients with\nPrimary + First Recurrence\nwith expression data\nn = {len(n_tp_r1_expr)}")
flow.node("D", f"Patients with glioblastoma\nn = {len(n_gbm)}")
flow.node("E", f"Patients with IDH-wildtype glioblastoma\nn = {len(n_idhwt_gbm)}")

# Arrows + number excluded at each step
flow.edge("A", "B",
    label=f"−{len(n_all - n_tp_r1)} excluded")

flow.edge("B", "C",
    label=f"−{len(n_tp_r1 - n_tp_r1_expr)} excluded")

flow.edge("C", "D",
    label=f"−{len(n_tp_r1_expr - n_gbm)} excluded")

flow.edge( "D", "E",
    label=f"−{len(n_gbm - n_idhwt_gbm)} excluded")

flow.attr(dpi="300")

flow.render(
    "../figures/glass_patient_selection",
    format="png",
    cleanup=True
)

flow

- Plot `glass_patient_selection.png` exported to figures directory.

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica Neue', 'Helvetica', 'Arial', 'sans-serif']

In [ ]:
# noinspection PyTypeChecker
plt.figure(figsize=(8, 8))

venn3(
    (tmz, radiotherapy, other_systemic),
    set_labels=(
        "TMZ",
        "Radiotherapy",
        "Other systemic treatment"
    )
)

plt.title(
    "Treatment overlap in IDH-wildtype GBM patients",
    fontsize=14,
    fontweight="bold",
    y=1.03
)
plt.savefig(
    "../figures/treatment_overlap_venn.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

- Plot `treatment_overlap_venn.png` exported to figures directory.

# **Gene expression processing and paired analysis**

## 1. Finalise the study cohort
- For further analysis `glass_idhwt_gbm_R1_pairs` will be used
- All patients have two samples
    - `PRIMARY_SAMPLE_ID`
    - `FIRST_RECURRENCE_SAMPLE_ID`

In [ ]:
print("Number of patients in final cohort:", glass_idhwt_gbm_primary_R1_pairs["PATIENT_ID"].nunique())
print("Number of samples in final cohort:", (glass_idhwt_gbm_primary_R1_pairs["PRIMARY_SAMPLE_ID"].nunique() + glass_idhwt_gbm_primary_R1_pairs["FIRST_RECURRENCE_SAMPLE_ID"].nunique()))

- 206 gene expression samples will be needed

## 2. Prepare the gene expression matrix - `glass_idhwt_gbm_primary_R1_TPM.tsv`
- Only keep gene expression data for samples that are part of cohort

In [ ]:
sample_ids = pd.concat([
    glass_idhwt_gbm_primary_R1_pairs["PRIMARY_SAMPLE_ID"],
    glass_idhwt_gbm_primary_R1_pairs["FIRST_RECURRENCE_SAMPLE_ID"],
])

glass_idhwt_gbm_primary_R1_TPM = data_mrna_seq_tpm.loc[
    :, data_mrna_seq_tpm.columns.isin(sample_ids)
].copy()

In [ ]:
glass_idhwt_gbm_primary_R1_TPM.iloc[:5, :5]

In [ ]:
glass_idhwt_gbm_primary_R1_TPM.to_csv(
    "../data/processed/glass_idhwt_gbm_primary_R1_TPM.tsv",
    sep="\t",
    index=True)

- `glass_idhwt_gbm_primary_R1_TPM.tsv` exported to /data/processed directory.

## 3. Perform initial quality control of gene expression data - `glass_expression_QC.xlsx`
- Check:
    - Number of genes in expression matrix
    - Duplicate gene IDs
    - Missing TPM values
    - Negative TPM values
    - Number of genes with 0 expression
    - Do all samples have valid expression  data?
- Sample summary table: `glass_expression_QC.xlsx`
    - Mean TPM
    - Median TPM
    - Number of genes with 0 expression
    - Maximum TPM

In [ ]:
glass_idhwt_gbm_primary_R1_TPM.iloc[:5, :5]

In [ ]:
n_genes = glass_idhwt_gbm_primary_R1_TPM.index.nunique()
n_genes_duplicate_ID = int(glass_idhwt_gbm_primary_R1_TPM.index.duplicated().sum())
duplicate_genes = glass_idhwt_gbm_primary_R1_TPM.loc[glass_idhwt_gbm_primary_R1_TPM.index.duplicated(keep = False)].copy()
n_tpm_missing = glass_idhwt_gbm_primary_R1_TPM.isna().sum().sum()
n_tpm_neg = (glass_idhwt_gbm_primary_R1_TPM < 0).sum().sum()

zero_gene_ids = glass_idhwt_gbm_primary_R1_TPM.index[
    (glass_idhwt_gbm_primary_R1_TPM == 0).all(axis=1)]
n_genes_zero = len(zero_gene_ids)

In [ ]:
print("Total number of gene in expression matrix:", n_genes)
print("Number of duplicate gene symbols:", n_genes_duplicate_ID)
print("Number of missing TPM values:",  n_tpm_missing)
print("Number of negative TPM values:", n_tpm_neg)
print("Number of genes with 0 expression in all samples:", n_genes_zero)

In [ ]:
glass_expression_QC = pd.DataFrame({
    "n_missing": glass_idhwt_gbm_primary_R1_TPM.isna().sum(axis=0),
    "n_negative": (glass_idhwt_gbm_primary_R1_TPM < 0).sum(axis=0),
    "n_0": (glass_idhwt_gbm_primary_R1_TPM == 0).sum(axis=0),
    "n_expressed": (glass_idhwt_gbm_primary_R1_TPM > 0).sum(axis=0),
    "mean_TPM": glass_idhwt_gbm_primary_R1_TPM.mean(axis=0),
    "median_TPM": glass_idhwt_gbm_primary_R1_TPM.median(axis=0),
    "max_TPM": glass_idhwt_gbm_primary_R1_TPM.max(axis=0)
})

glass_expression_QC

In [ ]:
glass_expression_QC.to_excel(
    "../results/qc/glass_expression_QC.xlsx",
    index = True
)

- `glass_expression_QC.xlsx` exported to processed data directory.

## 4. Log-transform TPM data
- log2(TPM+1)

In [ ]:
glass_idhwt_gbm_primary_R1_log2TPM = np.log2(glass_idhwt_gbm_primary_R1_TPM+1)

glass_idhwt_gbm_primary_R1_log2TPM = pd.DataFrame(
    glass_idhwt_gbm_primary_R1_log2TPM,
    index=glass_idhwt_gbm_primary_R1_TPM.index,
    columns=glass_idhwt_gbm_primary_R1_TPM.columns)

In [ ]:
glass_idhwt_gbm_primary_R1_log2TPM.to_csv(
    "../data/processed/glass_idhwt_gbm_primary_R1_log2TPM.tvs",
    sep="\t",
    index=True)

- `glass_idhwt_gbm_primary_R1_log2TPM.tvs` exported to /data/processed directory.

## 5. Filter lowly expressed genes + duplicated gene IDs with different expression - `glass_gene_filter_summary.xlsx`,  `glass_idhwt_gbm_primary_R1_log2TPM_filtered.tvs`
- Only keep genes which have TPM ≥ 1 in at least 20% of samples

In [ ]:
duplicate_genes

In [ ]:
glass_idhwt_gbm_primary_R1_TPM_filtered = (glass_idhwt_gbm_primary_R1_TPM.loc[~glass_idhwt_gbm_primary_R1_TPM.index.isin(duplicate_genes.index)].copy())

In [ ]:
min_samples = math.ceil(0.20 * glass_idhwt_gbm_primary_R1_TPM_filtered.shape[1])

glass_idhwt_gbm_primary_R1_TPM_filtered = glass_idhwt_gbm_primary_R1_TPM_filtered.loc[
    (glass_idhwt_gbm_primary_R1_TPM_filtered >= 1).sum(axis=1) >= min_samples
]

In [ ]:
glass_idhwt_gbm_primary_R1_log2TPM_filtered = glass_idhwt_gbm_primary_R1_log2TPM.loc[glass_idhwt_gbm_primary_R1_log2TPM.index.isin(glass_idhwt_gbm_primary_R1_TPM_filtered.index)].copy()

In [ ]:
n_genes_before = glass_idhwt_gbm_primary_R1_log2TPM.index.nunique()
n_genes_after = glass_idhwt_gbm_primary_R1_log2TPM_filtered.index.nunique()
n_genes_excluded = n_genes_before - n_genes_after

glass_gene_filter_summary = pd.DataFrame({
    "n_genes_before_filtering": [n_genes_before],
    "n_genes_after_filtering": [n_genes_after],
    "n_genes_excluded": [n_genes_excluded],
})

glass_gene_filter_summary

In [ ]:
glass_gene_filter_summary.to_excel(
    "../results/qc/glass_gene_filter_summary.xlsx",
    index = False)

In [ ]:
glass_idhwt_gbm_primary_R1_log2TPM_filtered.to_csv(
    "../data/processed/glass_idhwt_gbm_primary_R1_log2TPM_filtered.tvs",
    sep="\t",
    index=True)

- Final `glass_gene_filter_summary.xlsx` and `glass_idhwt_gbm_primary_R1_log2TPM_filtered.tvs` exported to /data/processed directory.

## 6. Assess global gene expression patterns across samples - `PCA_primary_recurrence.png`

In [ ]:
pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2))])

glass_idhwt_gbm_primary_R1_pca = pca_pipeline.fit_transform(
    glass_idhwt_gbm_primary_R1_log2TPM_filtered.T)

In [ ]:
sample_types = [
    "Primary"
    if sample_id in glass_idhwt_gbm_primary_R1_pairs["PRIMARY_SAMPLE_ID"].values
    else "First Recurrence"
    for sample_id in glass_idhwt_gbm_primary_R1_log2TPM_filtered.columns]

plt.figure(figsize=(8, 8))
for sample_type, marker in [
    ("Primary", "o"),
    ("First Recurrence", "o")
]:
    mask = [sample_type == x for x in sample_types]
    plt.scatter(
        glass_idhwt_gbm_primary_R1_pca[mask, 0],
        glass_idhwt_gbm_primary_R1_pca[mask, 1],
        marker=marker,
        s=50,
        label=sample_type)
pca = pca_pipeline.named_steps["pca"]
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% variance)", fontweight="bold")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% variance)", fontweight="bold")
plt.title(
    "Principal Component Analysis of Cohort Expression Profiles",
    fontsize=16,
    fontweight="bold",
    loc='center',
    y=1.05)
plt.legend()
plt.tight_layout()
plt.savefig(
    "../figures/PCA_primary_recurrence.png",
    dpi=300,
    bbox_inches="tight")

plt.show()

- Plot `PCA_primary_recurrence.png` exported to figures directory.

## 7. Perform paired comparison of gene expression between primary and recurrent samples - `glass_primary_vs_R1_gene_statistics.xlsx`
- delta_expression = R1 expression – Primary expression

In [ ]:
primary = glass_idhwt_gbm_primary_R1_log2TPM_filtered[glass_idhwt_gbm_primary_R1_pairs["PRIMARY_SAMPLE_ID"]]
r1 = glass_idhwt_gbm_primary_R1_log2TPM_filtered[glass_idhwt_gbm_primary_R1_pairs["FIRST_RECURRENCE_SAMPLE_ID"]]
delta_expression = r1.to_numpy() - primary.to_numpy()

In [ ]:
genes = glass_idhwt_gbm_primary_R1_log2TPM_filtered.index

print("Number of genes:", delta_expression.shape[0])
print("Number of patient pairs:", delta_expression.shape[1])

print("Primary samples:", primary.shape[1])
print("R1 samples:", r1.shape[1])

assert primary.shape[1] == r1.shape[1]

print( "Duplicated Primary IDs:", primary.columns.duplicated().sum())

print("Duplicated R1 IDs:",r1.columns.duplicated().sum())

In [ ]:
genes = glass_idhwt_gbm_primary_R1_log2TPM_filtered.index

skewness :list[float] = []

for delta in delta_expression:
    valid_delta = delta[~np.isnan(delta)]

    if len(valid_delta) >= 3:
        skewness.append(skew(valid_delta))
    else:
        skewness.append(np.nan)

normality_check = pd.DataFrame({
    "GENE": genes,
    "SKEWNESS": skewness})

print(f"Genes: {len(genes)}")
print(f"Patient pairs: {delta_expression.shape[1]}")
print()
median_abs_skew = normality_check["SKEWNESS"].abs().median()

print(f"Median |skewness|: {median_abs_skew:.3f}")

abs_skewness = normality_check["SKEWNESS"].abs()
n_skew_gt_1 = int(np.count_nonzero(abs_skewness.to_numpy() > 1))
n_skew_gt_2 = int(np.count_nonzero(abs_skewness.to_numpy() > 2))

print(f"Genes with |skewness| > 1: {n_skew_gt_1}")
print(f"Genes with |skewness| > 2: {n_skew_gt_2}")

> **Methodological Note:** Following $\log_2$-transformation, paired expression differences exhibited low overall skewness across genes (median absolute skewness = 0.29). Combined with a sample size of $N = 103$ paired observations, this justified the use of parametric paired $t$-tests under the Central Limit Theorem.

In [ ]:
genes = glass_idhwt_gbm_primary_R1_log2TPM_filtered.index

n_pairs = np.sum(~np.isnan(delta_expression), axis=1)

mean_primary = np.nanmean(primary.to_numpy(), axis=1)

mean_r1 = np.nanmean(r1.to_numpy(), axis=1)

mean_delta = np.nanmean(delta_expression, axis=1)

median_delta = np.nanmedian(delta_expression, axis=1)

sd_delta = np.nanstd(delta_expression, axis=1, ddof=1)

n_pairs = np.asarray(n_pairs, dtype=int)

t_statistic = mean_delta / (sd_delta / np.sqrt(n_pairs))

t_statistic = np.asarray(t_statistic, dtype=float)

p_value = np.asarray(
    2 * t.sf(
        np.abs(t_statistic),
        df=n_pairs - 1
    ),
    dtype=float
)

invalid = (
        (n_pairs < 2) |
        ~np.isfinite(t_statistic))

t_statistic[invalid] = np.nan
p_value[invalid] = np.nan

glass_primary_vs_R1_gene_statistics = pd.DataFrame({
    "GENE": genes,
    "N_PAIRS": n_pairs,
    "MEAN_PRIMARY": mean_primary,
    "MEAN_R1": mean_r1,
    "MEAN_DELTA": mean_delta,
    "MEDIAN_DELTA": median_delta,
    "SD_DELTA": sd_delta,
    "T_STATISTIC": t_statistic,
    "P_VALUE": p_value
})

glass_primary_vs_R1_gene_statistics["FDR"] = np.nan
valid_p = glass_primary_vs_R1_gene_statistics["P_VALUE"].notna()

In [ ]:
glass_primary_vs_R1_gene_statistics.loc[valid_p, "FDR"] = (
    multipletests(
        glass_primary_vs_R1_gene_statistics.loc[valid_p, "P_VALUE"],
        method="fdr_bh"
    )[1])

In [ ]:
glass_primary_vs_R1_gene_statistics

In [ ]:
glass_primary_vs_R1_gene_statistics.to_excel(
    "../results/differential_expression/glass_primary_vs_R1_gene_statistics.xlsx",
    index=False
)

- `glass_primary_vs_R1_gene_statistics.xlsx` exported to /results/differential_expression directory.

## 8. Assess the consistency of expression changes
- FRACTION_UP = ratio of patients where R1 > Primary
- FRACTION_DOWN = ratio of patients where R1 < Primary

In [ ]:
expr = glass_idhwt_gbm_primary_R1_log2TPM_filtered
pairs = glass_idhwt_gbm_primary_R1_pairs
up_counts = pd.Series(0, index=expr.index, dtype=float)
down_counts = pd.Series(0, index=expr.index, dtype=float)
n_valid = pd.Series(0, index=expr.index, dtype=float)

In [ ]:
for _, row in pairs.iterrows():
    primary_id = row["PRIMARY_SAMPLE_ID"]
    r1_id = row["FIRST_RECURRENCE_SAMPLE_ID"]
    if primary_id not in expr.columns or r1_id not in expr.columns:
        continue
    primary = expr[primary_id]
    r1 = expr[r1_id]
    valid = primary.notna() & r1.notna()
    up_counts += ((r1 > primary) & valid)
    down_counts += ((r1 < primary) & valid)
    n_valid += valid

In [ ]:
fractions_up = up_counts / n_valid
fractions_down = down_counts / n_valid

glass_primary_vs_R1_gene_statistics["FRACTION_UP"] = (glass_primary_vs_R1_gene_statistics["GENE"].map(fractions_up))
glass_primary_vs_R1_gene_statistics["FRACTION_DOWN"] = (glass_primary_vs_R1_gene_statistics["GENE"].map(fractions_down))

In [ ]:
glass_primary_vs_R1_gene_statistics.head()

## 9. Generate an initial list of recurrence-associated genes
- Recurrence-UP genes:
    - FDR < 0.05
    - MEAN_DELTA ≥ 0.5
    - FRACTION_UP ≥ 0.65
- Recurrence-DOWN genes:
    - FDR < 0.05
    - MEAN_DELTA ≤ -0.5
    - FRACTION_DOWN ≥ 0.65

In [ ]:
glass_recurrence_UP_initial = glass_primary_vs_R1_gene_statistics[
    (glass_primary_vs_R1_gene_statistics["FDR"] < 0.05) &
    (glass_primary_vs_R1_gene_statistics["MEAN_DELTA"] >= 0.5) &
    (glass_primary_vs_R1_gene_statistics["FRACTION_UP"] >= 0.65)
].copy()

glass_recurrence_DOWN_initial = glass_primary_vs_R1_gene_statistics[
    (glass_primary_vs_R1_gene_statistics["FDR"] < 0.05) &
    (glass_primary_vs_R1_gene_statistics["MEAN_DELTA"] <= -0.5) &
    (glass_primary_vs_R1_gene_statistics["FRACTION_DOWN"] >= 0.65)
].copy()

In [ ]:
glass_recurrence_UP_initial.head()

In [ ]:
glass_recurrence_DOWN_initial.head()

In [ ]:
glass_recurrence_UP_initial.to_excel(
    "../results/recurrence/glass_recurrence_UP_initial.xlsx",
    index=False)

glass_recurrence_DOWN_initial.to_excel(
    "../results/recurrence/glass_recurrence_DOWN_initial.xlsx",
    index=False)

- `glass_recurrence_UP_initial.xlsx` and `glass_recurrence_DOWN_initial.xlsx` exported to /data/processed directory.

## 10. Generate initial result plots - `glass_recurrence_vulcano.png`, `glass_recurrence_heatmap.png` `glass_recurrence_paired_plots.png`
- Vulcano plot
- Heatmap
- Some top genes' paired plot

In [ ]:
#Vulcano plot
stats = glass_primary_vs_R1_gene_statistics.copy()
stats["GENE"] = glass_idhwt_gbm_primary_R1_log2TPM_filtered.index.values
stats["NEG_LOG10_FDR"] = -np.log10(stats["FDR"].clip(lower=1e-300))

up_genes = set(glass_recurrence_UP_initial["GENE"])
down_genes = set(glass_recurrence_DOWN_initial["GENE"])

stats["CATEGORY"] = "Other"
stats.loc[stats["GENE"].isin(up_genes), "CATEGORY"] = "Recurrence-UP"
stats.loc[stats["GENE"].isin(down_genes), "CATEGORY"] = "Recurrence-DOWN"

fig, ax = plt.subplots(figsize=(10, 8))
other = stats["CATEGORY"] == "Other"
up = stats["CATEGORY"] == "Recurrence-UP"
down = stats["CATEGORY"] == "Recurrence-DOWN"
ax.scatter(
    stats.loc[other, "MEAN_DELTA"],
    stats.loc[other, "NEG_LOG10_FDR"],
    s=12,
    alpha=0.5,
    label="Other")
ax.scatter(
    stats.loc[up, "MEAN_DELTA"],
    stats.loc[up, "NEG_LOG10_FDR"],
    s=35,
    alpha=0.8,
    label="Recurrence-UP")
ax.scatter(
    stats.loc[down, "MEAN_DELTA"],
    stats.loc[down, "NEG_LOG10_FDR"],
    s=35,
    alpha=0.8,
    label="Recurrence-DOWN")
ax.axvline(0.5, linestyle="--", linewidth=1)
ax.axvline(-0.5, linestyle="--", linewidth=1)
ax.axhline(-np.log10(0.05), linestyle="--", linewidth=1)
ax.set_xlabel("MEAN_DELTA")
ax.set_ylabel("-log10(FDR)")
ax.set_title("Primary vs First Recurrence")
ax.legend()
top_up = (
    stats[stats["CATEGORY"] == "Recurrence-UP"]
    .sort_values("FDR")
    .head(10))
top_down = (
    stats[stats["CATEGORY"] == "Recurrence-DOWN"]
    .sort_values("FDR")
    .head(10))
for _, row in pd.concat([top_up, top_down]).iterrows():
    ax.annotate(
        row["GENE"],
        (row["MEAN_DELTA"], row["NEG_LOG10_FDR"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=7)
plt.tight_layout()
plt.savefig(
    "../figures/glass_recurrence_volcano.png",
    dpi=300,
    bbox_inches="tight")

plt.show()

- Plot `glass_recurrence_vulcano.png` exported to figures directory.

In [ ]:
#Heatmap
top_up = (
    glass_recurrence_UP_initial
    .sort_values("MEAN_DELTA", ascending=False)
    .head(10)["GENE"]
    .tolist()
)
top_down = (
    glass_recurrence_DOWN_initial
    .sort_values("MEAN_DELTA", ascending=True)
    .head(10)["GENE"]
    .tolist()
)

heatmap_genes = top_up + top_down

heatmap_expr = glass_idhwt_gbm_primary_R1_log2TPM_filtered

ordered_samples = []

for _, row in glass_idhwt_gbm_primary_R1_pairs.iterrows():

    primary = row["PRIMARY_SAMPLE_ID"]
    recurrence = row["FIRST_RECURRENCE_SAMPLE_ID"]

    if primary in heatmap_expr.columns:
        ordered_samples.append(primary)

    if recurrence in heatmap_expr.columns:
        ordered_samples.append(recurrence)

ordered_samples = [
    sample
    for sample in ordered_samples
    if sample in heatmap_expr.columns
]

heatmap_expr = heatmap_expr.loc[heatmap_genes, ordered_samples]

heatmap_gene_order = [
    gene
    for gene in heatmap_genes
    if gene in heatmap_expr.index
]

heatmap_expr = heatmap_expr.loc[heatmap_gene_order]

def draw_curly_brace_right(
    ax, y_min, y_max, x_pos=1.08, w=0.015, color="black", label=""
):
    """Draws a right-facing curly brace '}' positioned at x_pos on the right side of the heatmap."""
    trans = ax.get_yaxis_transform()

    y1 = y_min + 0.1
    y2 = y_max + 0.9
    ymid = (y1 + y2) / 2.0
    r = min(0.5, (y2 - y1) / 4.0)

    verts = [
        (x_pos, y1),
        (x_pos, y1 + r / 2),
        (x_pos + w / 2, y1 + r / 2),
        (x_pos + w / 2, y1 + r),
        (x_pos + w / 2, ymid - r),
        (x_pos + w / 2, ymid - r / 2),
        (x_pos + w, ymid - r / 2),
        (x_pos + w, ymid),
        (x_pos + w, ymid + r / 2),
        (x_pos + w / 2, ymid + r / 2),
        (x_pos + w / 2, ymid + r),
        (x_pos + w / 2, y2 - r),
        (x_pos + w / 2, y2 - r / 2),
        (x_pos, y2 - r / 2),
        (x_pos, y2),
    ]

    codes = [
        mpath.Path.MOVETO,
        mpath.Path.CURVE4,
        mpath.Path.CURVE4,
        mpath.Path.CURVE4,
        mpath.Path.LINETO,
        mpath.Path.CURVE4,
        mpath.Path.CURVE4,
        mpath.Path.CURVE4,
        mpath.Path.CURVE4,
        mpath.Path.CURVE4,
        mpath.Path.CURVE4,
        mpath.Path.LINETO,
        mpath.Path.CURVE4,
        mpath.Path.CURVE4,
        mpath.Path.CURVE4,
    ]

    path = mpath.Path(verts, codes)
    patch = mpatches.PathPatch(
        path,
        facecolor="none",
        edgecolor=color,
        lw=2,
        transform=trans,
        clip_on=False,
    )
    ax.add_patch(patch)

    if label:
        ax.text(
            x_pos + w + 0.01,
            ymid,
            label,
            transform=trans,
            color=color,
            fontweight="bold",
            fontsize=10,
            va="center",
            ha="left",
        )

gene_direction = pd.Series("UP", index=top_up)
gene_direction = pd.concat([
    gene_direction,
    pd.Series(
        "DOWN",
        index=top_down
    )
])
gene_direction = gene_direction.loc[
    heatmap_expr.index
]

p = sns.clustermap(
    heatmap_expr,
    z_score=0,
    row_cluster=True,
    col_cluster=False,
    cmap="viridis",
    center=0,
    vmin=-2,
    vmax=2,
    figsize=(20, 10),
    dendrogram_ratio=(
        0.12,
        0.02,
    ),
    colors_ratio=0.02,
    cbar_pos=(1.05, 0.01, 0.025, 0.96),
    xticklabels=False,
    yticklabels=True,
)

for i in range(2, heatmap_expr.shape[1], 2):
    p.ax_heatmap.axvline(i, color="white", linewidth=2)

if p.dendrogram_row is None:
    raise RuntimeError("Row dendrogram was not created.")
reordered_genes = heatmap_expr.index[p.dendrogram_row.reordered_ind]

up_indices = [i for i, gene in enumerate(reordered_genes) if gene in top_up]
down_indices = [i for i, gene in enumerate(reordered_genes) if gene in top_down]

if up_indices:
    draw_curly_brace_right(
        p.ax_heatmap,
        min(up_indices),
        max(up_indices),
        x_pos=1.06,
        color="#808588",
        label="UP",
    )

if down_indices:
    draw_curly_brace_right(
        p.ax_heatmap,
        min(down_indices),
        max(down_indices),
        x_pos=1.06,
        color="#808588",
        label="DOWN",
    )

p.ax_heatmap.tick_params(axis="y", labelsize=7)
p.ax_heatmap.set_xlabel(
    f"Paired Samples: Primary | First recurrence (n={heatmap_expr.shape[1] // 2} pairs)",
    fontsize=15,
    fontweight="bold",
    labelpad=10
)
p.ax_heatmap.set_ylabel(
    f"Hugo Symbol",
    fontweight="bold",
    fontsize=15
)
p.ax_heatmap.set_title(
    f"Z-Score Expression of Top Recurrence-Regulated Genes",
    fontsize=25,
    fontweight="bold",
    y=1.03
)

plt.savefig(
    "../figures/glass_recurrence_heatmap.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

- Plot `glass_recurrence_heatmap.png` exported to figures directory.

In [ ]:
#Paired plot
expr=glass_idhwt_gbm_primary_R1_log2TPM_filtered

top_up_plot = (
    glass_recurrence_UP_initial
    .sort_values(
        ["FRACTION_UP", "MEAN_DELTA"],
        ascending=[False, False])
    .head(5)["GENE"]
    .tolist())

top_down_plot = (
    glass_recurrence_DOWN_initial
    .sort_values(
        ["FRACTION_DOWN", "MEAN_DELTA"],
        ascending=[False, True])
    .head(5)["GENE"]
    .tolist())

plot_genes = top_up_plot + top_down_plot


fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for ax, gene in zip(axes.flat, plot_genes):
    is_up = gene in top_up_plot
    for _, row in pairs.iterrows():
        primary_id = row["PRIMARY_SAMPLE_ID"]
        r1_id = row["FIRST_RECURRENCE_SAMPLE_ID"]
        if primary_id not in expr.columns or r1_id not in expr.columns:
            continue
        primary_value = expr.loc[gene, primary_id]
        r1_value = expr.loc[gene, r1_id]
        if pd.isna(primary_value) or pd.isna(r1_value):
            continue
        ax.plot(
            [0, 1],
            [primary_value, r1_value],
            marker="o",
            alpha=0.35,
            linewidth=0.8
        )
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Primary", "R1"])
    ax.set_title(gene, fontsize=9)
    ax.set_xlim(-0.1, 1.1)
    ax.grid(axis="y", alpha=0.2)

plt.suptitle(
    "Patient-level paired expression changes",
    fontsize=25,
    fontweight="bold",
    y=1.03)
plt.tight_layout()
plt.savefig(
    "../figures/glass_recurrence_paired_plots.png",
    dpi=300,
    bbox_inches="tight")

plt.show()

- Plot `glass_recurrence_paired_plots.png` exported to figures directory.

# **Identifying biological processes associated with these changes**
- Input data: `glass_primary_vs_R1_gene_statistics`

## 1. Pathway analysis: Biological processes associated with recurrence

Gene-level paired $t$-statistics were used to rank genes for pre-ranked gene set enrichment analysis (GSEA).
MSigDB Hallmark (2020) and Reactome (2022) gene sets were analysed using the multilevel algorithm implemented in GSEApy (v1.3.1).

Outputs:
- `glass_GSEA_Hallmark.xlsx`
- `glass_GSEA_Reactome.xlsx`

In [ ]:
glass_primary_vs_R1_gene_statistics = pd.read_excel(
    "../results/differential_expression/glass_primary_vs_R1_gene_statistics.xlsx", sheet_name=0)

### 1.1 Generate ranked gene list

Genes were ranked according to the t-statistic from the primary tumour vs.
R1 comparison. Genes with missing statistics were removed, duplicate gene
symbols were removed, and the remaining genes were ranked from highest to
lowest t-statistic.

In [ ]:
stats = glass_primary_vs_R1_gene_statistics.copy()

ranked_stats = (
    stats[["GENE", "T_STATISTIC"]]
    .dropna()
    .drop_duplicates("GENE")
    .sort_values("T_STATISTIC", ascending=False)
)

rnk = ranked_stats.set_index("GENE")["T_STATISTIC"]

In [ ]:
rnk.head()

In [ ]:
rnk.tail()

In [ ]:
print("Number of ranked genes:", len(rnk))

### 1.2 Preranked GSEA

In [ ]:
hallmark = gp.get_library(
    name="MSigDB_Hallmark_2020",
    organism="Human")

print(len(hallmark))
print(list(hallmark.keys())[:5])

In [ ]:
# noinspection bad-argument-type
gsea = gp.prerank(
    rnk=rnk,
    gene_sets=hallmark,
    outdir=None,
    seed=42,
    verbose=True,
    method='multilevel'
)

In [ ]:
hallmark_results = gsea.res2d[["Term", "NES", "NOM p-val", "FDR q-val"]].copy()
hallmark_results = (hallmark_results.rename(
    columns={"Term": "PATHWAY",
             "NOM p-val": "NOMINAL_P",
             "FDR q-val": "FDR",})
                    .sort_values("FDR"))

In [ ]:
reactome = gp.get_library(
    name="Reactome_2022",
    organism="Human")

print(len(reactome))
print(list(reactome.keys())[:5])

In [ ]:
# noinspection bad-argument-type
gsea = gp.prerank(
    rnk=rnk,
    gene_sets=reactome,
    outdir=None,
    seed=42,
    verbose=True,
    method='multilevel'
)

In [ ]:
reactome_results = gsea.res2d[["Term", "NES", "NOM p-val", "FDR q-val"]].copy()
reactome_results =  ((reactome_results.rename(
    columns={"Term": "PATHWAY",
             "NOM p-val": "NOMINAL_P",
             "FDR q-val": "FDR",}))
                     .sort_values("FDR"))

In [ ]:
hallmark_results.to_excel(
    "../results/GSEA/glass_GSEA_Hallmark.xlsx",
    index=False
)

In [ ]:
reactome_results.to_excel(
    "../results/GSEA/glass_GSEA_Reactome.xlsx",
    index=False
)

The resulting pathway-level statistics include:
- **NES:** normalized enrichment score
- **NOMINAL_P:** nominal enrichment p-value
- **FDR:** false discovery rate

Results exported as Excel files:
- `/results/GSEA/glass_GSEA_Hallmark.xlsx`
- `/results/GSEA/glass_GSEA_Reactome.xlsx`

## 2. Summarising pathway analysis results

In [ ]:
def get_top_pathways(dataframe, number=10) -> pd.DataFrame:
    sig = dataframe[dataframe["FDR"] < 0.05].copy()
    positive = (
        sig[sig["NES"] > 0]
        .sort_values("NES", ascending=False)
        .head(number))
    negative = (
        sig[sig["NES"] < 0]
        .sort_values("NES", ascending=True)
        .head(number))
    return pd.concat([negative, positive])


hallmark_top = get_top_pathways(hallmark_results, number=10)
reactome_top = get_top_pathways(reactome_results, number=10)

In [ ]:
print("Significant:", (hallmark_results["FDR"] < 0.05).sum())
print("Positive significant:", ((hallmark_results["FDR"] < 0.05) & (hallmark_results["NES"] > 0)).sum())
print("Negative significant:", ((hallmark_results["FDR"] < 0.05) & (hallmark_results["NES"] < 0)).sum())

> **Methodological Note:** As only four positive Hallmark pathways reached statistical significance, all four were included in the analysis instead of selecting the top ten positive pathways.

In [ ]:
fig, axes = plt.subplots(
    2, 1,
    figsize=(12, 16))

datasets = [
    (hallmark_top, axes[0], "MSigDB Hallmark"),
    (reactome_top, axes[1], "Reactome")]

for df, ax, title in datasets:
    if df.empty:
        ax.text(
            0.5, 0.5,
            "No pathways with FDR < 0.05",
            ha="center",
            va="center",
            transform=ax.transAxes)
        ax.set_title(title)
        continue
    df = df.sort_values("NES")
    df["NEG_LOG10_FDR"] = -np.log10(
        df["FDR"].clip(lower=1e-300))
    y = range(len(df))
    colors = ['#d62728' if nes > 0 else '#1f77b4' for nes in df["NES"]]
    scatter = ax.scatter(
        df["NES"],
        y,
        s=df["NEG_LOG10_FDR"] * 30 + 20,
        c=colors,
        alpha=0.8)
    ax.axvline(
        0,
        color="gray",
        linestyle="--",
        linewidth=1)
    ax.set_yticks(list(y))
    ax.set_yticklabels(
        df["PATHWAY"],
        fontsize=8)
    ax.set_xlabel("Normalized Enrichment Score (NES)", fontweight="bold")
    ax.set_ylabel("Pathway", fontweight="bold")
    ax.set_title(title, pad=15)
    ax.grid(
        axis="x",
        alpha=0.2)
    handles, labels = scatter.legend_elements(
        prop="sizes",
        alpha=0.8,
        num=4,
        func=lambda s: (s - 20) / 30,
        color=scatter.get_facecolors()[0])
    ax.legend(
        handles,
        labels,
        title="-log10(FDR)",
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=False,
        labelspacing=2.5,
        handletextpad=1.0)

plt.suptitle(
    "Top Positively and Negatively Enriched Hallmark and Reactome Pathways\nBased on Differential Expression Between Primary and Recurrent Tumors\n(FDR < 0.05)",
    fontsize=20,
    fontweight="bold",
    y=0.98)

plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.savefig(
    "../figures/glass_GSEA_top_pathways.png",
    dpi=300,
    bbox_inches="tight")
plt.show()

In [ ]:
def format_pvalue(val):
    val = float(val)
    if val < 0.000001:
        return "<0.000001"
    return f"{val:.6f}"

def format_nes(val):
    return f"{float(val):.6f}"

In [ ]:
column_formatters = {
    "NES": format_nes,
    "NOMINAL_P": format_pvalue,
    "FDR": format_pvalue
}

In [ ]:
print("TOP HALLMARK PATHWAYS")
print(hallmark_top[["PATHWAY", "NES", "NOMINAL_P", "FDR"]].to_string(index=False))

In [ ]:
print("TOP REACTOME PATHWAYS")
print(reactome_top[["PATHWAY", "NES", "NOMINAL_P", "FDR"]].to_string(index=False))

In [ ]:
print("TOP HALLMARK PATHWAYS FORMATTED TO DECIMALS")
print(hallmark_top[["PATHWAY", "NES", "NOMINAL_P", "FDR"]].to_string(index=False, formatters=column_formatters))

In [ ]:
print("TOP REACTOME PATHWAYS FORMATTED TO 6 DECIMALS")
print(reactome_top[["PATHWAY", "NES", "NOMINAL_P", "FDR"]].to_string(index=False, formatters=column_formatters))

### 2.1. Biological relevance of top pathways:
**Overview:** Recurrent samples show a strong shift toward a less proliferative transcriptional state, with marked suppression of cell-cycle, DNA-repair, and genome-maintenance programs. In contrast, neuronal/synaptic and interferon-associated signatures are enriched in recurrent samples.

#### Key downregulated pathways
- **Cell cycle and proliferation:** Strong suppression of E2F Targets (NES: -3.16, FDR: <0.000001), G2-M Checkpoint (NES: -2.89, FDR: <0.000001), Myc Targets V1 (NES: -2.75, FDR: <0.000001), and Mitotic Spindle (NES: -1.65, FDR: 0.000404).
- **DNA repair and genome maintenance:** DNA Repair (NES: -1.91, FDR: 0.000004) is supported by Reactome enrichment of Chromosome Maintenance (NES: -2.62, FDR: <0.000001), Activation Of ATR In Response To Replication Stress (NES: -2.48, FDR: <0.000001), Homology Directed Repair (NES: -2.47, FDR: <0.000001) and other related pathways (all FDR: <0.000001).
- **Other pathways:** Notch Signaling (NES: -1.84, FDR: 0.003364), TGF-beta Signaling (NES: -1.68, FDR: 0.006668), and Oxidative Phosphorylation (NES: -1.67, FDR: 0.000312) are also reduced.

#### Key upregulated pathways
- **Signalling and neuronal programs:** Reactome shows strong enrichment of Amine Ligand-Binding Receptors (NES: 2.62, FDR: 0.000007), Serotonin Neurotransmitter Release (NES: 2.56, FDR: 0.000034), Neuronal System (NES: 2.42, FDR: <0.000001), and Transmission Across Chemical Synapses (NES: 2.22, FDR: <0.000001).
- **Other enriched programs:** KRAS Signaling Dn (NES: 2.02, FDR: 0.000010), Myogenesis (1.91, 0.000011), Interferon Alpha Response (NES: 1.80, FDR: 0.000852), and Heme Metabolism (NES: 1.48, FDR: 0.004748).

#### Biological Implication
- **Reduced proliferation:** The dominant feature of recurrence is suppression of E2F/Myc-driven proliferation, G2/M progression, mitosis, and DNA-repair/genome-maintenance programs.
- **Altered signalling and cellular programs:** Recurrent samples show enrichment of interferon-associated and neuronal/synaptic signatures, together with changes in metabolic and signalling pathways.
- **Interpretation:** The overall profile is consistent with a **low-proliferative recurrent state**. A senescence-like phenotype is a possible interpretation, but would require independent validation and should not be concluded from GSEA alone.


## 3. Assessing the stability of recurrence-associated genes using bootstrap analysis

In [ ]:
up_initial = glass_recurrence_UP_initial["GENE"]
down_initial = glass_recurrence_DOWN_initial["GENE"]
bootstrap_genes = pd.Index(up_initial.tolist() + down_initial.tolist()).drop_duplicates()
original_direction = pd.Series("UP", index=up_initial)
original_direction = pd.concat([original_direction, pd.Series("DOWN", index=down_initial)])

print("Initial UP genes:", len(up_initial))
print("Initial DOWN genes:", len(down_initial))

In [ ]:
len(genes) == delta_expression.shape[0]

In [ ]:
bootstrap_up_counts = pd.Series(
    0,
    index=bootstrap_genes,
    dtype=int)
bootstrap_down_counts = pd.Series(
    0,
    index=bootstrap_genes,
    dtype=int)
bootstrap_same_direction_counts = pd.Series(
    0,
    index=bootstrap_genes,
    dtype=int)

rng = np.random.default_rng(42)

n_patients = delta_expression.shape[1]
n_genes = delta_expression.shape[0]

print(f"Genes: {n_genes}")
print(f"Patient pairs: {n_patients}")
print(f"Original recurrence genes: {len(bootstrap_genes)}")
genes = glass_idhwt_gbm_primary_R1_log2TPM_filtered.index

original_up = bootstrap_genes[original_direction.loc[bootstrap_genes].eq("UP")]
original_down = bootstrap_genes[original_direction.loc[bootstrap_genes].eq("DOWN")]

for b in range(1000):
    sampled_idx = rng.integers(
        0,
        n_patients,
        size=n_patients)

    boot_delta = delta_expression[:, sampled_idx]

    n = np.sum(~np.isnan(boot_delta), axis=1)

    mean_delta = np.nanmean(
        boot_delta,
        axis=1)

    sd_delta = np.nanstd(
        boot_delta,
        axis=1,
        ddof=1)

    t_statistic = (mean_delta/(sd_delta/np.sqrt(n)))

    p_values = np.full(n_genes,np.nan)

    valid_tests = ((n >= 2) & np.isfinite(t_statistic))

    p_values[valid_tests] = (2 * t.sf(np.abs(t_statistic[valid_tests]), df=n[valid_tests] - 1))

    fdr = np.full(n_genes,np.nan)

    valid_p = np.isfinite(p_values)

    fdr[valid_p] = multipletests(
        p_values[valid_p],
        method="fdr_bh"
    )[1]

    fraction_up = np.divide(
    np.sum((boot_delta > 0) & ~np.isnan(boot_delta), axis=1),
    n,
    out=np.full(n.shape, np.nan, dtype=float),
    where=n > 0
    )

    fraction_down = np.divide(
    np.sum((boot_delta < 0) & ~np.isnan(boot_delta), axis=1),
    n,
    out=np.full(n.shape, np.nan, dtype=float),
    where=n > 0
    )

    boot_up = ((fdr < 0.05) &
          (mean_delta >= 0.5) &
          (fraction_up >= 0.65))
    boot_down = ((fdr < 0.05) &
            (mean_delta <= -0.5) &
            (fraction_down >= 0.65))

    up_genes = genes[boot_up]
    down_genes = genes[boot_down]

    selected_up = bootstrap_genes.intersection(pd.Index(up_genes))
    selected_down = bootstrap_genes.intersection(pd.Index(down_genes))

    same_up = original_up.intersection(pd.Index(up_genes))
    same_down = original_down.intersection(pd.Index(down_genes))

    bootstrap_up_counts.loc[selected_up] += 1
    bootstrap_down_counts.loc[selected_down] += 1

    bootstrap_same_direction_counts.loc[same_up] += 1
    bootstrap_same_direction_counts.loc[same_down] += 1

    if (b + 1) % 100 == 0:
        print(f"Bootstrap {b + 1}/1000 completed")

## 4. Determining bootstrap selection frequency
- BOOTSTRAP_UP_FREQUENCY = n_up_count/1000
- BOOTSTRAP_DOWN_FREQUENCY = n_down_count/1000
- BOOTSTRAP_SAME_DIRECTION_FREQUENCY = n_same_direction/1000

In [ ]:
bootstrap_stability = pd.DataFrame({
    "GENE": bootstrap_genes,
    "BOOTSTRAP_UP_FREQUENCY": bootstrap_up_counts.loc[bootstrap_genes].values / 1000,
    "BOOTSTRAP_DOWN_FREQUENCY": bootstrap_down_counts.loc[bootstrap_genes].values / 1000,
    "BOOTSTRAP_SAME_DIRECTION_FREQUENCY": bootstrap_same_direction_counts.loc[bootstrap_genes].values / 1000
})

In [ ]:
original_stats = pd.concat([
    glass_recurrence_UP_initial[
        ["GENE",
         "MEAN_DELTA",
         "FDR",
         "FRACTION_UP",
         "FRACTION_DOWN"]],
    glass_recurrence_DOWN_initial[
        ["GENE",
         "MEAN_DELTA",
         "FDR",
         "FRACTION_UP",
         "FRACTION_DOWN"]]], ignore_index=True)

original_stats = original_stats.rename(columns={
    "MEAN_DELTA": "ORIGINAL_MEAN_DELTA",
    "FDR": "ORIGINAL_FDR",
    "FRACTION_UP": "ORIGINAL_FRACTION_UP",
    "FRACTION_DOWN": "ORIGINAL_FRACTION_DOWN"})

In [ ]:
glass_recurrence_bootstrap_stability = (
    original_stats
    .merge(
        bootstrap_stability,
        on="GENE",
        how="left" ))

glass_recurrence_bootstrap_stability["ORIGINAL_DIRECTION"] = (glass_recurrence_bootstrap_stability["GENE"].map(original_direction))

In [ ]:
glass_recurrence_bootstrap_stability.head(20)

In [ ]:
glass_recurrence_bootstrap_stability = (
    glass_recurrence_bootstrap_stability
    .sort_values(
        "BOOTSTRAP_SAME_DIRECTION_FREQUENCY",
        ascending=False))

glass_recurrence_bootstrap_stability.to_excel(
    "../results/recurrence/glass_recurrence_bootstrap_stability.xlsx",
    index=False)

- `glass_recurrence_bootstrap_stability.xlsx` exported to `/results/recurrence/`

## 5. Defining the recurrence signature
### Gene-level differential expression and recurrence classification

For each gene $g$ and patient $i$, the paired expression difference between the first recurrence (R1) and primary tumour was calculated as:

$$
\Delta_{ig} = X_{ig}^{R1} - X_{ig}^{Primary}
$$

where $X_{ig}^{R1}$ and $X_{ig}^{Primary}$ denote the expression of gene $g$ in the R1 and primary tumour samples, respectively.

For each gene, $N_g$ denotes the number of patients with non-missing paired measurements.

#### Fraction of patients with increased expression at recurrence

$$
\mathrm{FRACTION\_UP}_g =
\frac{
\sum_{i=1}^{N_g} \mathbf{1}(\Delta_{ig} > 0)
}{
N_g
}
$$

#### Fraction of patients with decreased expression at recurrence

$$
\mathrm{FRACTION\_DOWN}_g =
\frac{
\sum_{i=1}^{N_g} \mathbf{1}(\Delta_{ig} < 0)
}{
N_g
}
$$

where $\mathbf{1}(\cdot)$ is the indicator function, equal to 1 when the specified condition is true and 0 otherwise.

The mean paired expression difference was calculated as:

$$
\overline{\Delta}_g =
\frac{1}{N_g}
\sum_{i=1}^{N_g} \Delta_{ig}
$$

### Recurrence-UP genes

A gene was classified as **Recurrence-UP** if all the following criteria were satisfied:

- **FDR < 0.05**
- **Mean difference $\overline{\Delta}_g \geq 0.5$**
- **FRACTION_UP $\geq 0.65$**

### Recurrence-DOWN genes

A gene was classified as **Recurrence-DOWN** if all the following criteria were satisfied:

- **FDR < 0.05**
- **Mean difference $\overline{\Delta}_g \leq -0.5$**
- **FRACTION_DOWN $\geq 0.65$**

### Bootstrap stability

Bootstrap stability was assessed using 1,000 patient-level bootstrap samples. In each bootstrap iteration, 103 patients were sampled with replacement, with the primary and R1 samples of each patient retained as a paired observation. For each bootstrap sample, gene-level paired differences, statistical test statistics, and FDR values were recalculated, and the same criteria used for the original Recurrence-UP and Recurrence-DOWN gene lists were applied.

For each gene, the **bootstrap same-direction frequency** was calculated as:

$$
\mathrm{BOOTSTRAP\_SAME\_DIRECTION\_FREQUENCY}_g =
\frac{B_g}{B}
$$

where $B = 1000$ is the total number of bootstrap iterations and $B_g$ is the number of bootstrap iterations in which the gene was identified in the same direction as in the original analysis.

A gene was considered **bootstrap-stable** if:

$$
\mathrm{BOOTSTRAP\_SAME\_DIRECTION\_FREQUENCY}_g \geq 0.70
$$

In [ ]:
STABILITY_THRESHOLD = 0.70

consensus = glass_recurrence_bootstrap_stability[
    glass_recurrence_bootstrap_stability[
        "BOOTSTRAP_SAME_DIRECTION_FREQUENCY"
    ] >= STABILITY_THRESHOLD
    ].copy()

consensus_up = consensus[
    consensus["ORIGINAL_DIRECTION"] == "UP"
    ].copy()

consensus_down = consensus[
    consensus["ORIGINAL_DIRECTION"] == "DOWN"
    ].copy()
n_original_up = len(glass_recurrence_UP_initial)
n_original_down = len(glass_recurrence_DOWN_initial)

n_consensus_up = len(consensus_up)
n_consensus_down = len(consensus_down)

n_consensus = len(consensus)
n_original = len(bootstrap_genes)

print("\nConsensus signature retention:")
print(f"UP:   {n_consensus_up}/{n_original_up} "f"({100 * n_consensus_up / n_original_up:.1f}%)")
print(f"DOWN: {n_consensus_down}/{n_original_down} "f"({100 * n_consensus_down / n_original_down:.1f}%)")
print(f"Overall: {n_consensus}/{n_original} "f"({100 * n_consensus / n_original:.1f}%)")

> **Summary Note:** Patient-level bootstrap analysis demonstrated that 168 of 315 (53.3%) Recurrence-UP genes and 97 of 220 (44.1%) Recurrence-DOWN genes were consistently selected in the same direction in at least 70% of bootstrap samples. Overall, 265 of 535 (49.5%) initially identified recurrence-associated genes met the predefined stability criterion.

In [ ]:
stability = glass_recurrence_bootstrap_stability.copy()

stability["OPPOSITE_DIRECTION_FREQUENCY"] = np.where(
    stability["ORIGINAL_DIRECTION"] == "UP",
    stability["BOOTSTRAP_DOWN_FREQUENCY"],
    stability["BOOTSTRAP_UP_FREQUENCY"]
)

consensus_inspection = stability.loc[
    stability["BOOTSTRAP_SAME_DIRECTION_FREQUENCY"] >= 0.70
].copy()

consensus_inspection[
    [
        "GENE",
        "ORIGINAL_DIRECTION",
        "BOOTSTRAP_SAME_DIRECTION_FREQUENCY",
        "OPPOSITE_DIRECTION_FREQUENCY"
    ]
].sort_values(
    "OPPOSITE_DIRECTION_FREQUENCY",
    ascending=False
).head(20)

In [ ]:
for threshold in [0.05, 0.10, 0.20, 0.30]:
    opposite_freq = consensus_inspection["OPPOSITE_DIRECTION_FREQUENCY"]
    n = int(np.count_nonzero(opposite_freq.to_numpy() >= threshold))
    print(f"Opposite-direction frequency >= {threshold:.0%}: "
          f"{n}/{len(consensus_inspection)}")

> **Summary Note:**  None of the 265 consensus genes showed evidence of meaningful direction switching during bootstrap resampling. For all consensus genes, opposite-direction selection occurred in <5% of bootstrap samples, supporting the stability of the UP/DOWN direction assignment.

In [ ]:
consensus_up.to_excel(
    "../results/recurrence/"
    "glass_recurrence_UP_consensus.xlsx",
    index=False)

consensus_down.to_excel(
    "../results/recurrence/"
    "glass_recurrence_DOWN_consensus.xlsx",
    index=False)

- `glass_recurrence_UP_consensus.xlsx` and `glass_recurrence_DOWN_consensus.xlsx` exported to `/results/recurrence/`

## 6. Creating the Recurrence Score
- Rank all genes within each sample
- rank(pct=True) gives percentile ranks from approximately 0 to 1.
- Higher expression → higher percentile.
- UP_SCORE = mean percentile rank of Consensus Recurrence-UP genes in sample
- DOWN_SCORE = mean percentile rank of Consensus Recurrence-DOWN genes in sample
- Uses frozen consensus signatures from section 5.
- High recurrence score = expression characteristic of R1 GBM
- Low recurrence score = expression characteristic of TP GBM

In [ ]:
def calculate_recurrence_score(expr, up_genes, down_genes):
    """Calculates sample-wise Recurrence Scores using gene percentile ranks.

    Parameters:
    -----------
    expr : pd.DataFrame
        Gene expression matrix with genes as rows and samples as columns.
    up_genes : list-like
        Gene symbols for consensus UP signature.
    down_genes : list-like
        Gene symbols for consensus DOWN signature.

    Returns:
    --------
    pd.Series
        Recurrence Score indexed by sample ID.
    """
    valid_up = [g for g in up_genes if g in expr.index]
    valid_down = [g for g in down_genes if g in expr.index]
    percentile_ranks = expr.rank(axis=0, method="average", pct=True)
    up_score = percentile_ranks.loc[valid_up].mean(axis=0)
    down_score = percentile_ranks.loc[valid_down].mean(axis=0)

    return pd.DataFrame({
        "UP_SCORE": up_score,
        "DOWN_SCORE": down_score,
        "RECURRENCE_SCORE": up_score - down_score
    })

In [ ]:
consensus_up_genes = pd.Index(consensus_up["GENE"]).intersection(expr.index)
consensus_down_genes = pd.Index(consensus_down["GENE"]).intersection(expr.index)

print(f"Consensus UP genes available: {len(consensus_up_genes)}")
print(f"Consensus DOWN genes available: {len(consensus_down_genes)}")

In [ ]:
scores_df = calculate_recurrence_score(glass_idhwt_gbm_primary_R1_log2TPM_filtered, consensus_up_genes, consensus_down_genes)
scores_df = scores_df.reset_index().rename(columns={"index": "SAMPLE_ID"})

In [ ]:
pairs = glass_idhwt_gbm_primary_R1_pairs
sample_info:pd.DataFrame = pd.concat([
    pairs[["PATIENT_ID", "PRIMARY_SAMPLE_ID"]].rename(columns={"PRIMARY_SAMPLE_ID": "SAMPLE_ID"}).assign(SAMPLE_TYPE="Tumor Primary"),
    pairs[["PATIENT_ID", "FIRST_RECURRENCE_SAMPLE_ID"]].rename(columns={"FIRST_RECURRENCE_SAMPLE_ID": "SAMPLE_ID"}).assign(SAMPLE_TYPE="First Recurrence")
], ignore_index=True)

In [ ]:
glass_recurrence_scores = sample_info.merge(scores_df, on="SAMPLE_ID", how="inner")
glass_recurrence_scores_full_cohort = glass_recurrence_scores[
    ["PATIENT_ID", "SAMPLE_ID", "SAMPLE_TYPE", "UP_SCORE", "DOWN_SCORE", "RECURRENCE_SCORE"]
].copy()

In [ ]:
glass_recurrence_scores_full_cohort.head()

In [ ]:
glass_recurrence_scores_full_cohort.to_excel(
    "../results/recurrence/glass_recurrence_scores_full_cohort.xlsx",
    index=False
)

- `glass_recurrence_scores_full_cohort.xlsx` exported to `/results/recurrence` directory.

In [ ]:
print(f"Recurrence scores calculated for "f"{len(glass_recurrence_scores)} samples.")

In [ ]:
glass_recurrence_scores[["UP_SCORE", "DOWN_SCORE", "RECURRENCE_SCORE"]].describe()

## 7. Initial assessment of the Recurrence Score in the full GLASS cohort

- DELTA_RECURRENCE_SCORE = SCORE_R1 – SCORE_PRIMARY

In [ ]:
glass_recurrence_scores.head()

In [ ]:
score_pairs = glass_recurrence_scores.pivot(
    index="PATIENT_ID",
    columns="SAMPLE_TYPE",
    values="RECURRENCE_SCORE"
)

score_pairs["DELTA_RECURRENCE_SCORE"] = (
    score_pairs["First Recurrence"] -
    score_pairs["Tumor Primary"]
)

In [ ]:
score_pairs.head()

In [ ]:
sns.kdeplot(np.array(score_pairs["DELTA_RECURRENCE_SCORE"]))

plt.title("Distribution of Paired Recurrence Score Differences")
plt.xlabel("R1 − Primary Recurrence Score")
plt.ylabel("Density")
plt.show()

In [ ]:
delta_skew_rs = float(skew(score_pairs["DELTA_RECURRENCE_SCORE"], nan_policy="omit"))

print("Patient pairs:", glass_recurrence_scores["PATIENT_ID"].nunique())
print(f"Skewness:{float(delta_skew_rs):.3f}")

> **Methodological Note:** The paired recurrence score differences exhibited minimal skewness (skewness = -0.045). Given the near-symmetric distribution of the differences and the sample size of $N = 103$ paired observations, the paired $t$-test was considered appropriate. The sample size also provides robustness to moderate deviations from normality through the Central Limit Theorem.

In [ ]:
mean_delta_rs = score_pairs["DELTA_RECURRENCE_SCORE"].mean()
median_delta_rs = score_pairs["DELTA_RECURRENCE_SCORE"].median()

t_statistic_rs, p_value_rs = ttest_rel(
    score_pairs["First Recurrence"],
    score_pairs["Tumor Primary"]
)

valid_rs = (
    score_pairs["First Recurrence"].notna()
    & score_pairs["Tumor Primary"].notna()
)

r1_higher_rs = (
    score_pairs.loc[valid_rs, "First Recurrence"]
    > score_pairs.loc[valid_rs, "Tumor Primary"]
).copy()

n_r1_higher_rs = score_pairs.loc[r1_higher_rs].index.nunique()
n_pairs_rs = int(valid_rs.sum())

fraction_r1_higher_rs = n_r1_higher_rs / n_pairs_rs
percent_r1_higher_rs = (
    score_pairs.loc[valid_rs, "First Recurrence"]
    > score_pairs.loc[valid_rs, "Tumor Primary"]
).mean()

In [ ]:
print("Recurrence Score paired comparison")
print(f"Mean delta: {mean_delta_rs:.4f}")
print(f"Median delta: {median_delta_rs:.4f}")
print(f"Paired t-test: t = {float(t_statistic_rs):.3f}, p = {float(p_value_rs):.3e}, {format_pvalue(p_value_rs)}")

print(
    f"R1 > Primary: "
    f"{n_r1_higher_rs}/{n_pairs_rs} "
    f"({100 * percent_r1_higher_rs:.1f}%)"
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for _, row in score_pairs.iterrows():
    ax.plot(
        ["Tumor Primary", "First Recurrence"],
        [row["Tumor Primary"], row["First Recurrence"]],
        marker="o",
        linewidth=0.8,
        alpha=0.35
    )

ax.set_ylabel("Recurrence Score")
ax.set_title("Recurrence Score: Tumor Primary vs First Recurrence", fontweight="bold", y=1.03)
ax.text(
    0.5,
    0.02,
    f"Mean Δ = {mean_delta_rs:.3f}, "
    f"paired p = {format_pvalue(p_value_rs)}",
    transform=ax.transAxes,
    ha="center",
    bbox=dict(
            boxstyle="round,pad=0.3", facecolor="white", edgecolor="none", alpha=0.8
        )
)
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()

plt.savefig(
    "../figures/glass_recurrence_score_paired.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

> **Summary Note:** The Recurrence Score increased substantially from Primary to first recurrence across the GLASS cohort (mean Δ = +0.223; median Δ = +0.227). The increase was statistically significant in the paired analysis (paired t-test, t = 9.249, p = 3.74 × 10⁻¹⁵), with 82/103 patients (79.6%) showing a higher Recurrence Score in their first recurrent samples than in their primary tumour sample. These findings support the ability of the rank-based Recurrence Score to capture a transcriptional program associated with tumour recurrence.

## 8. Out-of-fold validation using 5-fold cross-validation
- Is it possible to recognise recurrence in patients that were not part of creating the recurrence signature?
- out-of-fold recurrence score

In [ ]:
expr = glass_idhwt_gbm_primary_R1_log2TPM_filtered
pairs = glass_idhwt_gbm_primary_R1_pairs
genes = expr.index
n_patients = len(pairs)

In [ ]:
glass_idhwt_gbm_primary_R1_pairs.head()

In [ ]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_scores = []

assert len(pairs) == 103
assert pairs["PATIENT_ID"].nunique() == 103
assert pairs["PATIENT_ID"].is_unique

for fold, (train_idx, test_idx) in enumerate(
    kf.split(pairs),
    start=1
):
    train_idx = np.asarray(train_idx, dtype=int)
    test_idx = np.asarray(test_idx, dtype=int)

    print(f"\nFold {fold}/5")
    print(f"Training patients: {len(train_idx)}")
    print(f"Test patients: {len(pd.DataFrame(test_idx))}")

    # -------------------------
    # TRAINING SET
    # -------------------------

    train_pairs = pairs.iloc[train_idx]

    primary_ids = train_pairs["PRIMARY_SAMPLE_ID"].tolist()
    r1_ids = train_pairs["FIRST_RECURRENCE_SAMPLE_ID"].tolist()

    primary_train = expr.loc[:, primary_ids]
    r1_train = expr.loc[:, r1_ids]

    delta_train = (
        r1_train.to_numpy() -
        primary_train.to_numpy()
    )

    n_train_genes = delta_train.shape[0]

    n = np.sum(
        ~np.isnan(delta_train),
        axis=1
    )

    mean_delta = np.nanmean(
        delta_train,
        axis=1
    )

    sd_delta = np.nanstd(
        delta_train,
        axis=1,
        ddof=1
    )

    t_statistic = (
        mean_delta /
        (sd_delta / np.sqrt(n))
    )

    p_values = np.full(
        n_train_genes,
        np.nan,
        dtype=float
    )

    valid_tests = (
        (n >= 2) &
        np.isfinite(t_statistic)
    )

    p_values[valid_tests] = (
        2 *
        t.sf(
            np.abs(t_statistic[valid_tests]),
            df=n[valid_tests] - 1
        )
    )

    fdr = np.full(
        n_train_genes,
        np.nan,
        dtype=float
    )

    valid_p = np.isfinite(p_values)

    fdr[valid_p] = multipletests(
        p_values[valid_p],
        method="fdr_bh"
    )[1]

    fraction_up = np.divide(
        np.sum(
            (delta_train > 0) &
            ~np.isnan(delta_train),
            axis=1
        ),
        n,
        out=np.full(
            n.shape,
            np.nan,
            dtype=float
        ),
        where=n > 0
    )

    fraction_down = np.divide(
        np.sum(
            (delta_train < 0) &
            ~np.isnan(delta_train),
            axis=1
        ),
        n,
        out=np.full(
            n.shape,
            np.nan,
            dtype=float
        ),
        where=n > 0
    )

    train_up = (
        (fdr < 0.05) &
        (mean_delta >= 0.5) &
        (fraction_up >= 0.65)
    )

    train_down = (
        (fdr < 0.05) &
        (mean_delta <= -0.5) &
        (fraction_down >= 0.65)
    )

    up_genes = genes[train_up]
    down_genes = genes[train_down]

    print(f"Training UP genes: {len(up_genes)}")
    print(f"Training DOWN genes: {len(down_genes)}")

    # -------------------------
    # TEST SET
    # -------------------------

    test_pairs = pairs.iloc[test_idx]

    test_sample_ids = (
        test_pairs["PRIMARY_SAMPLE_ID"].tolist()
        +
        test_pairs["FIRST_RECURRENCE_SAMPLE_ID"].tolist()
    )

    test_expr = expr.loc[:, test_sample_ids]

    test_percentile_ranks = test_expr.rank(
        axis=0,
        method="average",
        pct=True
    )

    up_score = test_percentile_ranks.loc[
        test_percentile_ranks.index.intersection(
            up_genes
        )
    ].mean(axis=0)

    down_score = test_percentile_ranks.loc[
        test_percentile_ranks.index.intersection(
            down_genes
        )
    ].mean(axis=0)

    recurrence_score = (
        up_score -
        down_score
    )

    fold_scores = pd.DataFrame({
        "SAMPLE_ID": test_sample_ids,
        "UP_SCORE": up_score.to_numpy(),
        "DOWN_SCORE": down_score.to_numpy(),
        "RECURRENCE_SCORE": recurrence_score.to_numpy(),
        "FOLD": fold
    })

    fold_sample_info: pd.DataFrame = pd.concat([
        test_pairs.loc[:, ["PATIENT_ID", "PRIMARY_SAMPLE_ID"]
        ].rename({"PRIMARY_SAMPLE_ID": "SAMPLE_ID"},
                 axis="columns"
        ).assign(SAMPLE_TYPE="Tumor Primary"),
        test_pairs.loc[:, ["PATIENT_ID", "FIRST_RECURRENCE_SAMPLE_ID"]
        ].rename({"FIRST_RECURRENCE_SAMPLE_ID": "SAMPLE_ID"},
                 axis="columns",
        ).assign(SAMPLE_TYPE="First Recurrence")
    ],ignore_index=True
    )

    fold_scores = fold_sample_info.merge(
        fold_scores,
        on="SAMPLE_ID",
        how="left",
        validate="one_to_one"
    )

    oof_scores.append(fold_scores)

In [ ]:
glass_recurrence_scores_out_of_fold: pd.DataFrame = pd.concat(
    oof_scores,
    ignore_index=True
)

In [ ]:
glass_recurrence_scores_out_of_fold.head()

In [ ]:
print(f"\nOut-of-fold scores calculated for "
      f"{len(glass_recurrence_scores_out_of_fold)} samples "
      f"from {n_patients} patients.")

print(glass_recurrence_scores_out_of_fold[
        ["UP_SCORE",
         "DOWN_SCORE",
         "RECURRENCE_SCORE"]].describe())

In [ ]:
glass_recurrence_scores_out_of_fold.to_excel(
    "../results/recurrence/glass_recurrence_scores_out_of_fold.xlsx",
    index=False
)

- `glass_recurrence_scores_out_of_fold.xlsx` exported to `/results/recurrence` directory.

## 9. Assessing the out-of-fold Recurrence Score
- OOF_DELTA_SCORE = OOF_SCORE_R1 – OOF_SCORE_PRIMARY

In [ ]:
oof_pairs = pd.DataFrame(glass_recurrence_scores_out_of_fold.pivot(
    index="PATIENT_ID",
    columns="SAMPLE_TYPE",
    values="RECURRENCE_SCORE"
))

oof_pairs["OOF_DELTA_SCORE"] = (
    oof_pairs["First Recurrence"] -
    oof_pairs["Tumor Primary"]
)

In [ ]:
sns.kdeplot(np.array(oof_pairs["OOF_DELTA_SCORE"]))

plt.title("Distribution of Paired Out-of-Fold Recurrence Score Differences", y=1.03)
plt.xlabel("R1 − Primary Recurrence Score")
plt.ylabel("Density")
plt.show()

In [ ]:
oof_delta_skew = float(skew(oof_pairs["OOF_DELTA_SCORE"], nan_policy="omit"))

print("Patient pairs:", glass_recurrence_scores_out_of_fold["PATIENT_ID"].nunique())
print(f"Skewness:{float(oof_delta_skew):.3f}")

> **Methodological Note:** The paired out-of-fold recurrence score differences exhibited minimal skewness (skewness = -0.027). Given the near-symmetric distribution of the differences and the sample size of $N = 103$ paired observations, the paired $t$-test was considered appropriate. The sample size also provides robustness to moderate deviations from normality through the Central Limit Theorem.

In [ ]:
mean_delta_oof = oof_pairs["OOF_DELTA_SCORE"].mean()
median_delta_oof = oof_pairs["OOF_DELTA_SCORE"].median()

t_statistic_oof, p_value_oof = ttest_rel(
    oof_pairs["First Recurrence"],
    oof_pairs["Tumor Primary"]
)

valid_oof = (
    oof_pairs["First Recurrence"].notna()
    & oof_pairs["Tumor Primary"].notna()
)

r1_higher_oof = (
    oof_pairs.loc[valid_oof, "First Recurrence"]
    > oof_pairs.loc[valid_oof, "Tumor Primary"]
).copy()

n_r1_higher_oof = oof_pairs.loc[r1_higher_oof].index.nunique()
n_pairs_oof = int(valid_oof.sum())

fraction_r1_higher_oof = n_r1_higher_oof / n_pairs_oof
percent_r1_higher_oof = (
    oof_pairs.loc[valid_oof, "First Recurrence"]
    > oof_pairs.loc[valid_oof, "Tumor Primary"]
).mean()

In [ ]:
t_statistic_oof = float(t_statistic_oof)
p_value_oof = float(p_value_oof)

print("Out-of-fold Recurrence Score evaluation")
print(f"Patients: {len(oof_pairs)}")
print(f"Mean OOF delta: {mean_delta_oof:.4f}")
print(f"Median OOF delta: {median_delta_oof:.4f}")
print(
    f"Paired t-test: "
f"t = {t_statistic_oof:.3f}, "
f"p = {p_value_oof:.3e} {format_pvalue(p_value_oof)}")

print(
    f"R1 > Primary: "
    f"{n_r1_higher_oof}/{n_pairs_oof} "
    f"({100 * percent_r1_higher_oof:.1f}%)"
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for _, row in oof_pairs.iterrows():
    ax.plot(
        ["Tumor Primary", "First Recurrence"],
        [row["Tumor Primary"], row["First Recurrence"]],
        marker="o",
        linewidth=0.8,
        alpha=0.35
    )

ax.set_ylabel("Out-of-fold Recurrence Score")
ax.set_title("Out-of-fold Recurrence Score: Primary vs R1", fontweight="bold", y=1.03)

ax.text(
    0.5,
    0.02,
    f"Mean Δ = {mean_delta_oof:.3f}, "
    f"paired p = {format_pvalue(p_value_oof)}",
    transform=ax.transAxes,
    ha="center",
    bbox=dict(
            boxstyle="round,pad=0.3", facecolor="white", edgecolor="none", alpha=0.8
        )
)

plt.tight_layout()

plt.savefig(
    "../figures/glass_recurrence_score_out_of_fold_paired.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
roc_data = glass_recurrence_scores_out_of_fold.copy()

roc_data["LABEL"] = (
        roc_data["SAMPLE_TYPE"] == "First Recurrence"
).astype(int)

oof_auc = roc_auc_score(
    roc_data["LABEL"],
    roc_data["RECURRENCE_SCORE"]
)

print(f"OOF ROC AUC: {oof_auc:.3f}")

> **Summary Note:** The Recurrence Score was consistently higher at first recurrence than in the corresponding Primary samples (mean Δ = 0.176; median Δ = 0.168; paired t = 8.004, p = 2.00 × 10⁻¹²), with 77.7% of patients showing an increased score at R1. The out-of-fold Recurrence Score also discriminated between Primary and R1 samples with an ROC AUC of 0.770. These findings support the ability of the Recurrence Signature to identify recurrence-associated transcriptional patterns in patients who were not used to construct the corresponding signature.

## 10. Sensitivity analysis in patients receiving standard therapy

- Treatment selection and interval mapping
    - Assessable window: Only treatments initiated strictly between primary and recurrent sampling dates (`PRIMARY_DATE` < `START_DATE` $\le$ `RECURRENCE_DATE`) were included.
    - Invalid or non-chronological dates were marked `pd.NA`.
    - Standard therapies:
        - Radiotherapy (RT): Any radiation initiated within the window.
        - Temozolomide (TMZ): Standard monotherapy (`temozolomide`, `temodar`, `tmz`).
        - Concurrent TMZ+RT: Simultaneous initiation of TMZ and RT on the exact same date (`START_DATE`).
        - Other Systemic: Includes all non-TMZ chemotherapy, modified dosing (e.g., LD TMZ), combination regimens (e.g., TMZ + Lomustine), and completely different regimens (e.g., Bevacizumab, PCV).

In [ ]:
print("From cohort, has received TMZ treatment:", len(tmz))
print("From cohort, has received radiotherapy:", len(radiotherapy))
print("From cohort, has received concurrent TMZ treatment and radiotherapy:", len(concurrent_tmz))
print("From cohort, has received other systemic treatments:", len(other_systemic))

In [ ]:
expr_matrix = glass_idhwt_gbm_primary_R1_log2TPM_filtered.copy()

scores_series = calculate_recurrence_score(expr_matrix, consensus_up_genes, consensus_down_genes)

scores_df = pd.DataFrame(
    {"SAMPLE_ID": scores_series.index, "RECURRENCE_SCORE": scores_series["RECURRENCE_SCORE"]}
)

scores_df = scores_df.merge(
    glass_recurrence_scores_full_cohort[["SAMPLE_ID", "PATIENT_ID"]].drop_duplicates(),
    on="SAMPLE_ID",
    how="left",
)

In [ ]:
subgroups = {
    "Standard TMZ + RT":  (tmz & radiotherapy),
    "Standard TMZ + RT (No Other Systemic)": (tmz & radiotherapy - other_systemic)
}

paired_dfs = {}

for title, patient_ids in subgroups.items():
    sub_df = treatment_pairs.loc[treatment_pairs["PATIENT_ID"].isin(patient_ids)].copy()

    samples:pd.DataFrame = pd.concat(
        [sub_df[["PATIENT_ID", "PRIMARY_SAMPLE_ID"]
            ].rename(columns={"PRIMARY_SAMPLE_ID": "SAMPLE_ID"}
            ).assign(SAMPLE_TYPE="Tumor Primary"),
            sub_df[["PATIENT_ID", "FIRST_RECURRENCE_SAMPLE_ID"]
            ].rename(columns={"FIRST_RECURRENCE_SAMPLE_ID": "SAMPLE_ID"}
            ).assign(SAMPLE_TYPE="First Recurrence"),
        ],ignore_index=True,)

    merged = samples.merge(
        scores_df,
        on=["PATIENT_ID", "SAMPLE_ID"],
        how="left",)

    paired = (
        merged
        .pivot(
            index="PATIENT_ID",
            columns="SAMPLE_TYPE",
            values="RECURRENCE_SCORE",).reset_index())

    paired["DELTA_RECURRENCE_SCORE"] = (
        paired["First Recurrence"]
        - paired["Tumor Primary"])

    paired_dfs[title] = paired

In [ ]:
with pd.ExcelWriter(
    "../results/recurrence/glass_recurrence_score_standard_treatment.xlsx"
) as writer:
    for title, df in paired_dfs.items():
        sheet = (
            "Standard_TMZ_RT"
            if "No Other" not in title
            else "Standard_TMZ_RT_No_Other"
        )
        df.to_excel(writer, sheet_name=sheet, index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)

for ax, (title, paired) in zip(axes, paired_dfs.items()):
    mean_delta = paired["DELTA_RECURRENCE_SCORE"].mean()
    _, p_val = ttest_rel(paired["First Recurrence"], paired["Tumor Primary"])

    for _, row in paired.iterrows():
        ax.plot(
            ["Tumor Primary", "First Recurrence"],
            [row["Tumor Primary"], row["First Recurrence"]],
            marker="o",
            linewidth=0.8,
            alpha=0.35,
        )

    ax.set_ylabel("Recurrence Score")
    ax.set_title(f"{title}\n(n = {len(paired)})", fontweight="bold", y=1.03)

    ax.text(
        0.5,
        0.02,
        f"Mean Δ = {mean_delta:.3f}, paired p = {format_pvalue(p_val)}",
        transform=ax.transAxes,
        ha="center",
        bbox=dict(
            boxstyle="round,pad=0.3", facecolor="white", edgecolor="none", alpha=0.8
        ),
    )

plt.tight_layout()

plt.savefig(
    "../figures/glass_recurrence_score_standard_treatment.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

> **Summary Note:** Sensitivity analyses confirmed that the observed increase in Recurrence Score is not driven by non-standard treatment regimens.
>
> - **Standard TMZ + RT Cohort:** Recurrence Scores increased significantly from Primary to R1 (mean Δ = 227, paired t-test p < 0.000001).
> - **Standard TMZ + RT (No Other Systemic Therapy):** The Recurrence Score increase remained robust and statistically significant when excluding patients who received non-standard systemic agents (mean Δ = 229, paired t-test p < 0.000001).
>
> These results confirm that the consensus Recurrence Signature tracks intrinsic glioblastoma progression under standard treatment.

# **Can the transcriptional program associated with tumour recurrence be identified in glioblastoma cell lines?**

## 1. Solidifying Recurrence Score

In [ ]:
glass_recurrence_UP_consensus = consensus_up.copy()
glass_recurrence_DOWN_consensus = consensus_down.copy()

In [ ]:
glass_recurrence_signature_final = pd.DataFrame(pd.concat(
    [glass_recurrence_UP_consensus,
    glass_recurrence_DOWN_consensus],
    ignore_index=True
).rename(columns={
    "ORIGINAL_DIRECTION" : "DIRECTION",
    "ORIGINAL_MEAN_DELTA": "MEAN_DELTA",
    "ORIGINAL_FDR": "FDR"
}))

In [ ]:
glass_recurrence_signature_final = glass_recurrence_signature_final[
    ["GENE",
    "DIRECTION",
    "MEAN_DELTA",
    "FDR",
    "BOOTSTRAP_SAME_DIRECTION_FREQUENCY"]
].copy()

In [ ]:
glass_recurrence_signature_final.head()

In [ ]:
glass_recurrence_signature_final.to_excel(
    "../results/recurrence/glass_recurrence_signature_final.xlsx",
    index = False
)

## 2. Defining the final glioma cell-line cohort

### Load and filter sensitivity data
Checking if there's sensitivity data for all the expression data

### Load and filter expression data
* Model.cvs was used for identification of cell line oncotree lineages
* Source: https://depmap.org/portal/data_page/?tab=allData&releasename=DepMap%20Public%2026Q1&filename=Model.csv
* Two expression datasets were used:

In [ ]:
model = pd.read_csv(
    "../data/raw/annotation/Model.csv",
    index_col=0,
    low_memory=False,
)

model.index.name = "ModelID"

In [ ]:
model_metadata = (
    model[
        [
            "CellLineName",
            "StrippedCellLineName",
            "COSMICID",
            "CCLEName",
            "SangerModelID",
            "ModelIDAlias",
            "OncotreeLineage",
            "OncotreePrimaryDisease"
        ]
    ]
    .copy()
)

model_metadata.index.name = "ModelID"

In [ ]:
glioma_model_ids = model.index[
    model["OncotreePrimaryDisease"]
    .str.contains("glioma", case=False, na=False)
]

print("Total glioma cell lines in Model.csv:", len(glioma_model_ids))

#### DepMap - Log-transformed TPM (Transcripts Per Million) values at gene level derived from unstranded RNA-seq for all genes in human
- **Raw Data:** `OmicsExpressionTPMLogp1HumanAllGenes.csv`
- **Source:** [https://depmap.org/portal/download/all/?release=DepMap+Public+26Q1&file=OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv&tab=allData](https://depmap.org/portal/data_page/?tab=allData&releasename=DepMap%20Public%2026Q1&filename=OmicsExpressionTPMLogp1HumanAllGenes.csv)

In [ ]:
OmicsExpressionTPMLogp1HumanAllGenes = pd.read_csv(
    "../data/raw/DepMap/expression/OmicsExpressionTPMLogp1HumanAllGenes.csv",
    index_col=0,
)

gene = pd.read_csv(
    "../data/raw/annotation/Gene.csv",
    index_col=0,
    low_memory=False
)

In [ ]:
OmicsExpressionTPMLogp1HumanAllGenes.iloc[:5,:20]

In [ ]:
depmap_exp = OmicsExpressionTPMLogp1HumanAllGenes.drop(
    columns=[
        "SequencingID",
        "ModelConditionID",
        "IsDefaultEntryForMC",
        "IsDefaultEntryForModel",
    ]
).copy()

depmap_exp = depmap_exp.set_index("ModelID")
depmap_exp = depmap_exp.rename(columns=lambda x: re.sub(r"\s*\(.*?\)", "", x))

In [ ]:
gene_map = (
    gene
    .dropna(subset=["ensembl_gene_id", "symbol"])
    .drop_duplicates("ensembl_gene_id")
    .set_index("ensembl_gene_id")["symbol"]
)

In [ ]:
mapped = depmap_exp.columns.to_series().map(gene_map)

print("Total genes:", len(depmap_exp.columns))
print("Mapped:", mapped.notna().sum())
print("Unmapped:", mapped.isna().sum())

In [ ]:
duplicates = depmap_exp.columns[depmap_exp.columns.duplicated()]

print("Duplicate symbols:", len(duplicates))
print(duplicates.tolist())

In [ ]:
original_columns = depmap_exp.columns.to_series()

depmap_exp.columns = (
    original_columns
    .map(gene_map)
    .fillna(original_columns)
)

In [ ]:
depmap_exp.iloc[:5, :5]

In [ ]:
depmap_exp_metadata = model.reindex(depmap_exp.index)

depmap_glioma_ids = depmap_exp_metadata.index[
    depmap_exp_metadata["OncotreePrimaryDisease"]
    .str.contains("glioma", case=False, na=False)
]

depmap_glioma_exp = depmap_exp.loc[
    depmap_exp.index.isin(depmap_glioma_ids)
].copy()

print("DepMap total:", depmap_exp.index.nunique())
print("DepMap glioma:", depmap_glioma_exp.index.nunique())
print("DepMap expression variables:", depmap_glioma_exp.columns.nunique())

### DepMap expression data quality check

In [ ]:
depmap_expression_qc = {}

depmap_expression_qc["Dataset"] = (
    "OmicsExpressionTPMLogp1HumanAllGenes.csv"
)

depmap_expression_qc["Platform"] = (
    "Log-Transformed  RNA-seq"
)

depmap_expression_qc["Total_cell_lines"] = (
    depmap_exp.index.nunique()
)

depmap_expression_qc["Total_expression_variables"] = (
    depmap_exp.columns.nunique()
)

depmap_expression_qc["Minimum_expression"] = (
    np.nanmin(depmap_exp.to_numpy(dtype=float))
)

depmap_expression_qc["Maximum_expression"] = (
    np.nanmax(depmap_exp.to_numpy(dtype=float))
)

In [ ]:
missing_depmap_expression = glioma_model_ids.difference(depmap_exp.index)

print("Glioma cell lines according to Model.csv:", len(glioma_model_ids))
print("Glioma cell lines with DepMap expression:", depmap_glioma_exp.index.nunique())
print("Glioma cell lines missing DepMap expression:", len(missing_depmap_expression))

In [ ]:
depmap_missing_metadata = model.loc[
    missing_depmap_expression,
    [
        "CellLineName",
        "COSMICID",
        "OncotreeLineage",
        "OncotreePrimaryDisease",
        "OncotreeSubtype",
    ]
].copy()

In [ ]:
depmap_expression_available = depmap_exp.index.nunique()

depmap_expression_qc["Glioma_cohort_size"] = len(glioma_model_ids)

depmap_expression_qc["Glioma_with_DepMap_expression"] = (
    depmap_expression_available
)

depmap_expression_qc["Glioma_missing_DepMap_expression"] = (
    len(missing_depmap_expression)
)

depmap_expression_qc["Expression_variable_type"] = (
    "Gene symbols"
)

In [ ]:
depmap_expression_qc["Duplicate_ModelIDs"] = (
    depmap_exp.index.duplicated().sum()
)

In [ ]:
all_missing_profiles = depmap_exp.index[
    depmap_exp.isna().all(axis=1)
]

depmap_expression_qc["Completely_missing_profiles"] = (
    len(all_missing_profiles)
)

print(
    "Completely missing expression profiles:",
    all_missing_profiles.tolist()
)

In [ ]:
depmap_expression_qc["Duplicated_gene_variables"] = (
    depmap_exp.columns.duplicated().sum()
)

In [ ]:
depmap_qc_summary = pd.DataFrame(
    {
        "Metric": [
            "Dataset",
            "Platform",
            "Total DepMap cell lines",
            "Glioma cell lines (Model.csv)",
            "Glioma cell lines with DepMap expression",
            "Glioma cell lines missing DepMap expression",
            "Expression variables",
            "Expression variable type",
            "Duplicated cell-line IDs",
            "Completely missing expression profiles",
            "Duplicated expression variables",
            "Minimum expression value",
            "Maximum expression value",
        ],
        "Value": [
            depmap_expression_qc["Dataset"],
            depmap_expression_qc["Platform"],
            depmap_expression_qc["Total_cell_lines"],
            depmap_expression_qc["Glioma_cohort_size"],
            depmap_expression_qc["Glioma_with_DepMap_expression"],
            depmap_expression_qc["Glioma_missing_DepMap_expression"],
            depmap_expression_qc["Total_expression_variables"],
            depmap_expression_qc["Expression_variable_type"],
            depmap_expression_qc["Duplicate_ModelIDs"],
            depmap_expression_qc["Completely_missing_profiles"],
            depmap_expression_qc["Duplicated_gene_variables"],
            depmap_expression_qc["Minimum_expression"],
            depmap_expression_qc["Maximum_expression"],
        ],
    }
)

In [ ]:
depmap_qc_summary

In [ ]:
depmap_glioma = model.loc[
    model["OncotreePrimaryDisease"]
    .str.contains("glioma", case=False, na=False),
    ["CellLineName"]
].copy()


depmap_glioma = depmap_glioma.rename(
    columns={
        "CellLineName": "CELL_LINE_NAME"
    }
)

In [ ]:
depmap_glioma_qc = pd.DataFrame(
    index=depmap_glioma.index
)

depmap_glioma_qc.index.name = "ModelID"

depmap_glioma_qc["DEPMAP_EXPRESSION_AVAILABLE"] = (
    depmap_glioma_qc.index.isin(depmap_exp.index)
)

depmap_glioma_qc["CELL_LINE_NAME"] = (
    depmap_glioma["CELL_LINE_NAME"]
)

In [ ]:
duplicate_model_ids = (
    depmap_exp.index[
        depmap_exp.index.duplicated(keep=False)
    ]
    .unique()
)

In [ ]:
with pd.ExcelWriter(
    "../results/qc/depmap_expression_QC.xlsx"
) as writer:

    depmap_qc_summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False,
    )

    depmap_glioma_qc.to_excel(
        writer,
        sheet_name="Glioma_Cohort",
    )

    if len(missing_depmap_expression) > 0:
        pd.DataFrame(
            {"ModelID": missing_depmap_expression}
        ).to_excel(
            writer,
            sheet_name="Missing_Expression",
            index=False,
        )

    if len(duplicate_model_ids) > 0:
        pd.DataFrame(
            {"ModelID": duplicate_model_ids}
        ).to_excel(
            writer,
            sheet_name="Duplicate_Model_IDs",
            index=False,
        )

#### **GDSC1/GDSC2 RNA-normalized Affymetrix HGU-219 Expression Matrix:**
- As of 19/7/2026, the preprocessed expression data released by the Sanger Institute is not available on the original website anymore.
- This source was used to trace back the original link: https://github.com/BorgwardtLab/Kernelized-Rank-Learning/blob/master/URLs_to_GDSC_datasets.txt
- **Raw Data ArrayExpress Accession:** `E-MTAB-3610`
- **Preprocessed Data:** `data/raw/GDSC/expression/Cell_line_RMA_proc_basalExp.txt`
- **Original Source:** http://www.cancerrxgene.org/gdsc1000/GDSC1000_WebResources/Data/preprocessed/Cell_line_RMA_proc_basalExp.txt.zip
- **Currently Available Source:** https://web.archive.org/web/20260209143748/http://www.cancerrxgene.org/gdsc1000/GDSC1000_WebResources/Data/preprocessed/Cell_line_RMA_proc_basalExp.txt.zip

In [ ]:
gdsc_exp = pd.read_csv(
    "../data/raw/GDSC/expression/Cell_line_RMA_proc_basalExp.txt",
    sep="\t",
    low_memory=False,
)

In [ ]:
gdsc_gene_metadata = gdsc_exp[
    ["GENE_SYMBOLS", "GENE_title"]
].copy()

gdsc_gene_metadata["SOURCE"] = (
    gdsc_gene_metadata["GENE_title"]
    .str.extract(r"\[Source:([^;\]]+)", expand=False)
    .str.strip()
    .str.replace(r"\s+Symbol$", "", regex=True)
)

gdsc_gene_metadata["GENE_SYMBOL"] = (
    gdsc_gene_metadata["GENE_SYMBOLS"]
    .str.strip()
)

gdsc_gene_metadata = gdsc_gene_metadata[
    ["GENE_SYMBOL", "SOURCE"]
].drop_duplicates()

gdsc_gene_metadata.head()

In [ ]:
expression_columns = [
    col for col in gdsc_exp.columns
    if col not in {"GENE_SYMBOLS", "GENE_title"}
]

gdsc_exp[expression_columns] = (
    gdsc_exp[expression_columns]
    .apply(pd.to_numeric, errors="coerce")
)

In [ ]:
gdsc_exp = (
    gdsc_exp
    .dropna(subset=["GENE_SYMBOLS"])
    .drop(columns=["GENE_title"])
    .set_index("GENE_SYMBOLS")
    .T
)

gdsc_exp.index = (
    gdsc_exp.index
    .astype(str)
    .str.replace(r"^DATA\.", "", regex=True)
    .str.strip()
)

gdsc_exp.columns.name = None
gdsc_exp.index.name = "COSMICID"

In [ ]:
model_cosmic = (
    model[["COSMICID"]]
    .dropna()
    .copy()
)

model_cosmic["COSMICID"] = (
    model_cosmic["COSMICID"]
    .astype("Int64")
    .astype("string")
)

cosmic_model_counts = (
    model_cosmic
    .reset_index()
    .groupby("COSMICID")["ModelID"]
    .nunique()
)

ambiguous_cosmic_ids = cosmic_model_counts[
    cosmic_model_counts > 1
]

print(
    "COSMICIDs mapping to multiple ModelIDs:",
    len(ambiguous_cosmic_ids),
)

In [ ]:
cosmic_to_model = (
    model_cosmic
    .reset_index()
    .drop_duplicates("COSMICID")
    .set_index("COSMICID")["ModelID"]
)

gdsc_exp["ModelID"] = (
    pd.Series(gdsc_exp.index, index=gdsc_exp.index)
    .map(cosmic_to_model)
)

In [ ]:
gdsc_exp = (
    gdsc_exp
    .set_index("ModelID")
)

gdsc_exp.index.name = "ModelID"

### GDSC expression data quality check

In [ ]:
gdsc_expression_qc = {}

gdsc_expression_qc["Dataset"] = (
    "Cell_line_RMA_proc_basalExp"
)

gdsc_expression_qc["Platform"] = (
    "Affymetrix Human Genome U219"
)

gdsc_expression_qc["Total_cell_lines"] = (
    gdsc_exp.index.nunique()
)

gdsc_expression_qc["Total_expression_variables"] = (
    len(gdsc_exp.columns)
)

gdsc_expression_qc["Minimum_expression"] = (
    np.nanmin(gdsc_exp.to_numpy(dtype=float))
)

gdsc_expression_qc["Maximum_expression"] = (
    np.nanmax(gdsc_exp.to_numpy(dtype=float))
)

In [ ]:
gdsc_glioma_expression_ids = gdsc_exp.index.intersection(glioma_model_ids)

In [ ]:
missing_gdsc_expression = glioma_model_ids.difference(gdsc_exp.index)

print("Glioma cell lines according to Model.csv:", len(glioma_model_ids))
print("Glioma cell lines with GDSC expression:", len(gdsc_glioma_expression_ids))
print("Glioma cell lines missing GDSC expression:", len(missing_gdsc_expression))

In [ ]:
gdsc_glioma_exp = gdsc_exp.loc[
    gdsc_exp.index.isin(gdsc_glioma_expression_ids)
].copy()

In [ ]:
gdsc_missing_metadata = model.loc[
    missing_gdsc_expression,
    [
        "CellLineName",
        "COSMICID",
        "OncotreeLineage",
        "OncotreePrimaryDisease",
        "OncotreeSubtype",
    ]
].copy()

In [ ]:
gdsc_expression_available = gdsc_glioma_expression_ids.isin(
    gdsc_exp.index
)

gdsc_expression_qc["Glioma_cohort_size"] = len(gdsc_glioma_expression_ids)

gdsc_expression_qc["Glioma_with_expression"] = (
    gdsc_expression_available.sum()
)

gdsc_expression_qc["Glioma_missing_expression"] = (
    (~gdsc_expression_available).sum()
)

gdsc_expression_qc["Expression_variable_type"] = (
    "Gene symbols"
)

In [ ]:
gdsc_expression_qc["Duplicate_ModelIDs"] = (
    gdsc_exp.index.duplicated().sum()
)

In [ ]:
all_missing_profiles = gdsc_exp.index[
    gdsc_exp.isna().all(axis=1)
]

gdsc_expression_qc["Completely_missing_profiles"] = (
    len(all_missing_profiles)
)

print(
    "Completely missing expression profiles:",
    all_missing_profiles.tolist()
)

In [ ]:
gdsc_expression_qc["Duplicated_gene_variables"] = (
    gdsc_exp.columns.duplicated().sum()
)

In [ ]:
gdsc_qc_summary = pd.DataFrame(
    {
        "Metric": [
            "Dataset",
            "Platform",
            "Total cell lines",
            "Glioma cell lines",
            "Glioma cell lines with expression",
            "Glioma cell lines missing expression",
            "Expression variables",
            "Expression variable type",
            "Duplicated cell-line IDs",
            "Completely missing expression profiles",
            "Duplicated expression variables",
            "Minimum expression value",
            "Maximum expression value",
        ],
        "Value": [
            gdsc_expression_qc["Dataset"],
            gdsc_expression_qc["Platform"],
            gdsc_expression_qc["Total_cell_lines"],
            gdsc_expression_qc["Glioma_cohort_size"],
            gdsc_expression_qc["Glioma_with_expression"],
            gdsc_expression_qc["Glioma_missing_expression"],
            gdsc_expression_qc["Total_expression_variables"],
            gdsc_expression_qc["Expression_variable_type"],
            gdsc_expression_qc["Duplicate_ModelIDs"],
            gdsc_expression_qc["Completely_missing_profiles"],
            gdsc_expression_qc["Duplicated_gene_variables"],
            gdsc_expression_qc["Minimum_expression"],
            gdsc_expression_qc["Maximum_expression"],
        ],
    }
)

In [ ]:
gdsc_qc_summary

In [ ]:
gdsc_glioma = model.loc[
    model["OncotreePrimaryDisease"]
    .str.contains("glioma", case=False, na=False),
    ["CellLineName", "COSMICID"]
].copy()


gdsc_glioma = gdsc_glioma.rename(
    columns={
        "CellLineName": "CELL_LINE_NAME",
        "COSMICID": "GDSC_COSMIC_ID",
    }
)

In [ ]:
gdsc_glioma_qc = pd.DataFrame(
    index=gdsc_glioma.index
)

gdsc_glioma_qc.index.name = "ModelID"

gdsc_glioma_qc["GDSC_EXPRESSION_AVAILABLE"] = (
    gdsc_glioma_qc.index.isin(gdsc_exp.index)
)

gdsc_glioma_qc["CELL_LINE_NAME"] = (
    gdsc_glioma["CELL_LINE_NAME"]
)

gdsc_glioma_qc["GDSC_COSMIC_ID"] = (
    gdsc_glioma["GDSC_COSMIC_ID"]
)

In [ ]:
duplicate_model_ids = (
    gdsc_exp.index[
        gdsc_exp.index.duplicated(keep=False)
    ]
    .unique()
)

In [ ]:
with pd.ExcelWriter(
    "../results/qc/gdsc_expression_QC.xlsx"
) as writer:

    gdsc_qc_summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False,
    )

    gdsc_glioma_qc.to_excel(
        writer,
        sheet_name="Glioma_Cohort",
    )

    if len(missing_gdsc_expression) > 0:
        pd.DataFrame(
            {"ModelID": missing_gdsc_expression}
        ).to_excel(
            writer,
            sheet_name="Missing_Expression",
            index=False,
        )

    if len(duplicate_model_ids) > 0:
        pd.DataFrame(
            {"ModelID": duplicate_model_ids}
        ).to_excel(
            writer,
            sheet_name="Duplicate_Model_IDs",
            index=False,
        )

#### Drug sensitivity data
- AUC is normalised to 1:
- **CTD^2 Raw Data:** `data/raw/Drug_sensitivity_AUC_(CTD^2)_subsetted.csv`
- **DepMap Raw Data:** `data/raw/Drug_sensitivity_AUC_(PRISM_Repurposing_Secondary_Screen)_subsetted.csv`
- **GDSC1 Raw Data:** `data/raw/Drug_sensitivity_AUC_(Sanger_GDSC1)_subsetted.csv`
- **GDSC2 Raw Data:** `data/raw/Drug_sensitivity_AUC_(Sanger_GDSC2)_subsetted.csv`
- **Sources:**
    - https://depmap.org/portal/download/all/?release=Harmonized+CTD%5E2+25Q2&file=CTRPAUCMatrix.csv&tab=allData
    - https://depmap.org/portal/download/all/?release=Harmonized+PRISM+Repurposing+Secondary+Screen+25Q2&file=REPURPOSINGAUCMatrix.csv&tab=allData
    - https://depmap.org/portal/download/all/?release=Harmonized+GDSC+25Q2&file=GDSC1AUCMatrix.csv&tab=allData
    - https://depmap.org/portal/download/all/?release=Harmonized+GDSC+25Q2&file=GDSC2AUCMatrix.csv&tab=allData
- AUC isn't normalised:
- **CTRPv2 Raw Data:**
    -  `../data/raw/old/sensitivity/glioma/CTRPv2.2_2015_pub_CancerDisc_5_1210.zip`
        - `v22.data.auc_sensitivities.txt`
        - `v22.meta.per_cell_line.txt`
        - `v22.meta.per_compound.txt`
- **Source:** https://d1hp0ufb0xxisr.cloudfront.net/CTD2/Broad/CTRPv2.2_2015_pub_CancerDisc_5_1210.zip

In [ ]:
ctd2 = pd.read_csv(
    "../data/raw/DepMap/sensitivity/"
    "Drug_sensitivity_AUC_(CTD^2)_subsetted.csv",
    index_col=0,
)

prism = pd.read_csv(
    "../data/raw/DepMap/sensitivity/"
    "Drug_sensitivity_AUC_(PRISM_Repurposing_Secondary_Screen)_subsetted.csv",
    index_col=0,
)

gdsc1 = pd.read_csv(
    "../data/raw/GDSC/sensitivity/"
    "Drug_sensitivity_AUC_(Sanger_GDSC1)_subsetted.csv",
    index_col=0,
)

gdsc2 = pd.read_csv(
    "../data/raw/GDSC/sensitivity/"
    "Drug_sensitivity_AUC_(Sanger_GDSC2)_subsetted.csv",
    index_col=0,
)

#### Formatting CTRPv2 sensitivity data to match the others

In [ ]:
auc = pd.read_csv(
    "../data/raw/DepMap/sensitivity/CTRPv2.2_2015_pub_CancerDisc_5_1210/"
    "v22.data.auc_sensitivities.txt",
    sep="\t",
)

cell_metadata = pd.read_csv(
    "../data/raw/DepMap/sensitivity/CTRPv2.2_2015_pub_CancerDisc_5_1210/"
    "v22.meta.per_cell_line.txt",
    sep="\t",
)

compound_metadata = pd.read_csv(
    "../data/raw/DepMap/sensitivity/CTRPv2.2_2015_pub_CancerDisc_5_1210/"
    "v22.meta.per_compound.txt",
    sep="\t",
)

In [ ]:
def normalize_name(value):
    if pd.isna(value):
        return ""

    return re.sub(
        r"[^A-Z0-9]",
        "",
        str(value).upper(),
    )

In [ ]:
model_for_mapping = model_metadata.reset_index()

model_name_mapping = pd.concat(
    [
        model_for_mapping[["ModelID", "CellLineName"]].rename(
            columns={"CellLineName": "model_name"}
        ),
        model_for_mapping[["ModelID", "StrippedCellLineName"]].rename(
            columns={"StrippedCellLineName": "model_name"}
        ),
        model_for_mapping[["ModelID", "CCLEName"]]
        .assign(
            CCLEName=lambda x: x["CCLEName"]
            .fillna("")
            .str.split("_")
            .str[0]
        )
        .rename(columns={"CCLEName": "model_name"}),
    ],
    ignore_index=True,
)

model_name_mapping["normalized_name"] = (
    model_name_mapping["model_name"].map(normalize_name)
)

model_name_mapping = (
    model_name_mapping
    .loc[model_name_mapping["normalized_name"] != ""]
    .drop_duplicates(
        subset=["normalized_name", "ModelID"]
    )
)

model_name_mapping.head()

In [ ]:
ambiguous_names = (
    model_name_mapping
    .groupby("normalized_name")["ModelID"]
    .unique()
    .loc[lambda x: x.str.len() > 1]
)

print("Ambiguous cell-line names:", len(ambiguous_names))

if len(ambiguous_names) > 0:
    print(ambiguous_names)

In [ ]:
name_to_model = (
    model_name_mapping
    .groupby("normalized_name")["ModelID"]
    .agg(lambda x: list(x))
)


def choose_model_id(ccl_name):
    normalized_name = normalize_name(ccl_name)
    matches = name_to_model.get(normalized_name, [])

    if len(matches) == 1:
        return matches[0]

    return pd.NA


cell_metadata["ModelID"] = (
    cell_metadata["ccl_name"]
    .map(choose_model_id)
)

In [ ]:
print(
    "CTRPv2 profiles:",
    cell_metadata["index_ccl"].nunique()
)

print(
    "Mapped CTRPv2 profiles:",
    cell_metadata["ModelID"].notna().sum()
)

print(
    "Unmapped/ambiguous CTRPv2 profiles:",
    cell_metadata["ModelID"].isna().sum()
)

In [ ]:
ctrpv2_unresolved = (
    cell_metadata.loc[
        cell_metadata["ModelID"].isna(),
        ["index_ccl", "ccl_name"]
    ]
    .drop_duplicates()
    .copy()
)

print(
    "Unique unresolved CTRPv2 cell lines:",
    len(ctrpv2_unresolved)
)

In [ ]:
compound_metadata["compound_label"] = (
    compound_metadata["cpd_name"].str.upper()
    + " (CTRP:"
    + compound_metadata["master_cpd_id"]
        .astype("Int64")
        .astype("string")
    + ")"
)

if compound_metadata["compound_label"].duplicated().any():
    raise ValueError(
        "Compound labels are not unique."
    )

In [ ]:
ambiguous_ctrpv2_metadata = (
    cell_metadata
    .dropna(subset=["ModelID"])
    .groupby("ModelID")["ccl_name"]
    .nunique()
)

ambiguous_ctrpv2_metadata = (
    ambiguous_ctrpv2_metadata[
        ambiguous_ctrpv2_metadata > 1
    ]
)

print(
    "ModelIDs with multiple distinct CTRPv2 names:",
    len(ambiguous_ctrpv2_metadata)
)

In [ ]:
ctrpv2_metadata = (
    cell_metadata[
        ["ModelID", "ccl_name"]
    ]
    .dropna(subset=["ModelID"])
    .drop_duplicates()
    .sort_values("ModelID")
    .drop_duplicates(subset=["ModelID"], keep="first")
    .set_index("ModelID")
)

ctrpv2_metadata.index.name = "ModelID"

In [ ]:
auc = auc.merge(
    cell_metadata[["index_ccl", "ModelID"]],
    on="index_ccl",
    how="left",
    validate="many_to_one",
)

auc = auc.merge(
    compound_metadata[["index_cpd", "compound_label"]],
    on="index_cpd",
    how="left",
    validate="many_to_one",
)

In [ ]:
auc = auc.dropna(
    subset=["ModelID"]
)

In [ ]:
ctrpv2 = (
    auc
    .pivot_table(
        index="ModelID",
        columns="compound_label",
        values="area_under_curve",
        aggfunc="mean",
    )
    .sort_index()
    .sort_index(axis=1)
)

ctrpv2.columns.name = None
ctrpv2.index.name = "ModelID"

In [ ]:
ctrpv2.head()

In [ ]:
ctrpv2_metadata.head()

In [ ]:
print("\nCTRPv2 formatted:")
print("Cell lines:", ctrpv2.shape[0])
print("Compounds:", ctrpv2.shape[1])
print("Shape:", ctrpv2.shape)

In [ ]:
ctrpv2.to_csv(
    "../data/raw/DepMap/sensitivity/"
    "Drug_sensitivity_AUC_(CTRPv2.2)_wide.csv",
    index=True,
)

## 3. Glioma Cell Line Masterdoc

In [ ]:
cell_line_master = pd.DataFrame(
    index=model.index
)

cell_line_master.index.name = "ModelID"

In [ ]:
model_metadata = model_metadata.rename(
    columns={
        "CellLineName": "CELL_LINE_NAME",
        "StrippedCellLineName": "STRIPPED_CELL_LINE_NAME",
        "COSMICID": "GDSC_COSMIC_ID",
        "SangerModelID": "SANGER_MODEL_ID",
        "CCLEName": "CCLE_NAME",
        "ModelIDAlias": "DEPMAP_MODEL_ID_ALIAS"
    }
)

model_metadata = model_metadata.join(
    ctrpv2_metadata[["ccl_name"]].rename(
        columns={"ccl_name": "CTRPV2_NAME"}
    ),
    how="left",
    validate="one_to_one",
)

In [ ]:
databases = {
    "CTD2": ctd2,
    "PRISM": prism,
    "CTRPV2": ctrpv2,
    "GDSC1": gdsc1,
    "GDSC2": gdsc2,
}

cell_line_master = model_metadata.copy()

for name, df in databases.items():
    cell_line_master[f"{name}_AVAILABLE"] = (
        cell_line_master.index.isin(df.index)
    )

cell_line_master["GDSC_EXPRESSION_AVAILABLE"] = (
    cell_line_master.index.isin(gdsc_exp.index)
)

cell_line_master["DEPMAP_EXPRESSION_AVAILABLE"] = (
    cell_line_master.index.isin(depmap_exp.index)
)

cell_line_master["GDSC_EXPRESSION_AVAILABLE"] = (
    cell_line_master.index.isin(gdsc_exp.index)
)

cell_line_master["DEPMAP_EXPRESSION_AVAILABLE"] = (
    cell_line_master.index.isin(depmap_exp.index)
)

In [ ]:
gdsc_sensitivity_available = (cell_line_master["GDSC1_AVAILABLE"] | cell_line_master["GDSC2_AVAILABLE"])

depmap_sensitivity_available =  (cell_line_master["CTRPV2_AVAILABLE"]
    | cell_line_master["PRISM_AVAILABLE"]
    | cell_line_master["CTD2_AVAILABLE"])

gdsc_complete = gdsc_sensitivity_available & cell_line_master["GDSC_EXPRESSION_AVAILABLE"]
depmap_complete = depmap_sensitivity_available & cell_line_master["DEPMAP_EXPRESSION_AVAILABLE"]

either_complete = gdsc_complete | depmap_complete
both_complete = gdsc_complete & depmap_complete

print("GDSC has expression and sensitivity:", gdsc_complete.sum())
print("DepMap has expression and sensitivity:", depmap_complete.sum())
print("Has expression and sensitivity in either:", either_complete.sum())
print("Has expression and sensitivity in both:", both_complete.sum())

In [ ]:
availability_cols = [
    "GDSC1_AVAILABLE",
    "GDSC2_AVAILABLE",
    "CTD2_AVAILABLE",
    "CTRPV2_AVAILABLE",
    "PRISM_AVAILABLE",
    "GDSC_EXPRESSION_AVAILABLE",
    "DEPMAP_EXPRESSION_AVAILABLE",
]

metadata_cols = [
    "CELL_LINE_NAME",
    "STRIPPED_CELL_LINE_NAME",
    "GDSC_COSMIC_ID",
    "SANGER_MODEL_ID",
    "CCLE_NAME",
    "DEPMAP_MODEL_ID_ALIAS",
    "OncotreeLineage",
    "OncotreePrimaryDisease",
    "CTRPV2_NAME",
]

cell_line_master = (
    cell_line_master
    .groupby(level="ModelID")
    .agg({
        **{col: "any" for col in availability_cols},
        **{col: "first" for col in metadata_cols},
    })
)

In [ ]:
cell_line_master["GDSC_SENSITIVITY_AVAILABLE"] = (
    gdsc_sensitivity_available
)

cell_line_master["DEPMAP_SENSITIVITY_AVAILABLE"] = (
    depmap_sensitivity_available
)

cell_line_master["GDSC_COMPLETE"] = gdsc_complete
cell_line_master["DEPMAP_COMPLETE"] = depmap_complete
cell_line_master["EITHER_COMPLETE"] = either_complete
cell_line_master["BOTH_COMPLETE"] = both_complete

In [ ]:
cell_line_master = (
    cell_line_master
    .reset_index()
    .rename(columns={"ModelID": "DEPMAP_ID"})
)
cell_line_master = cell_line_master.set_index("CELL_LINE_NAME")

In [ ]:
column_order = [
    "STRIPPED_CELL_LINE_NAME",
    "DEPMAP_ID",
    "GDSC_COSMIC_ID",
    "SANGER_MODEL_ID",
    "CCLE_NAME",
    "CTRPV2_NAME",
    "DEPMAP_MODEL_ID_ALIAS",
    "OncotreeLineage",
    "OncotreePrimaryDisease",
    "GDSC1_AVAILABLE",
    "GDSC2_AVAILABLE",
    "CTD2_AVAILABLE",
    "CTRPV2_AVAILABLE",
    "PRISM_AVAILABLE",
    "GDSC_EXPRESSION_AVAILABLE",
    "DEPMAP_EXPRESSION_AVAILABLE",
]

cell_line_master = cell_line_master[column_order]
cell_line_master["NOTES"] = ""

In [ ]:
cell_line_master.to_excel(
    "../results/cell_lines/cell_line_master.xlsx"
)

In [ ]:
glioma_cell_line_master = cell_line_master[
    cell_line_master["OncotreePrimaryDisease"]
    .str.contains("glioma", case=False, na=False)
].copy()

In [ ]:
glioma_cell_line_master.to_excel(
    "../results/cell_lines/glioma_cell_line_master.xlsx"
)

- `cell_line_master.xlsx` and `glioma_cell_line_master.xlsx` saved to `/results/cell_lines`

## 4. Merging GLASS signature and DepMap expression data

In [ ]:
depmap_glioma_exp.columns = depmap_glioma_exp.columns.str.replace(r"\s*\(.*?\)", "", regex=True)

In [ ]:
GLASS_genes = glass_idhwt_gbm_primary_R1_log2TPM_filtered.index
DepMap_genes = depmap_glioma_exp.columns
consensus_UP_genes = pd.Index(glass_recurrence_UP_consensus["GENE"])
consensus_DOWN_genes = pd.Index(glass_recurrence_DOWN_consensus["GENE"])
consensus_genes = consensus_UP_genes.union(consensus_DOWN_genes)

In [ ]:
DepMap_genes

In [ ]:
depmap_glioma_exp.iloc[:5, :5:]

In [ ]:
glass_depmap_scoring_gene_universe = GLASS_genes.intersection(DepMap_genes)

In [ ]:
glass_depmap_common_gene_universe = pd.DataFrame(
    {
        "Metric": [
            "GLASS filtered genes",
            "DepMap genes",
            "Common genes",
            "Signature UP genes",
            "Signature DOWN genes available"
        ],
        "Value": [
            GLASS_genes.nunique(),
            DepMap_genes.nunique(),
            glass_depmap_scoring_gene_universe.nunique(),
            glass_depmap_scoring_gene_universe.intersection(consensus_UP_genes).nunique(),
            glass_depmap_scoring_gene_universe.intersection(consensus_DOWN_genes).nunique()
        ],
    }
)

In [ ]:
glass_depmap_common_gene_universe

In [ ]:
glass_depmap_common_gene_universe.to_csv(
    "../results/DepMap/glass_depmap_common_gene_universe.txt",
    index = False
)

- `glass_depmap_common_gene_universe.txt` exported to `results/DepMap/` directory.

## 5. Calculating DepMap Recurrence Score

In [ ]:
depmap_glioma_common_exp = depmap_glioma_exp[
    depmap_glioma_exp.columns.intersection(
        glass_depmap_scoring_gene_universe
    )
].copy()

depmap_glioma_common_exp = depmap_glioma_common_exp.T

In [ ]:
UP_common = pd.Index(depmap_glioma_common_exp.index.intersection(consensus_UP_genes))

In [ ]:
DOWN_common = pd.Index(depmap_glioma_common_exp.index.intersection(consensus_DOWN_genes))

In [ ]:
depmap_glioma_recurrence_scores = calculate_recurrence_score(depmap_glioma_common_exp, UP_common, DOWN_common)

In [ ]:
up_genes_present = pd.Index(UP_common).intersection(
    depmap_glioma_common_exp.index
)

n_up_genes = (
    depmap_glioma_common_exp
    .loc[up_genes_present, :]
    .notna()
    .sum(axis=0)
)

n_up_genes.index.name = "DEPMAP_ID"

depmap_glioma_recurrence_scores["N_UP_GENES"] = (
    n_up_genes.reindex(
        depmap_glioma_recurrence_scores.index
    )
)

down_genes_present = pd.Index(DOWN_common).intersection(
    depmap_glioma_common_exp.index
)

n_down_genes = (
    depmap_glioma_common_exp
    .loc[down_genes_present, :]
    .notna()
    .sum(axis=0)
)

n_down_genes.index.name = "DEPMAP_ID"

depmap_glioma_recurrence_scores["N_DOWN_GENES"] = (
    n_down_genes.reindex(
        depmap_glioma_recurrence_scores.index
    )
)


In [ ]:
depmap_glioma_recurrence_scores["CELL_LINE_NAME"] = model["CellLineName"].reindex(depmap_glioma_recurrence_scores.index)

depmap_glioma_recurrence_scores = depmap_glioma_recurrence_scores.reset_index()

depmap_glioma_recurrence_scores = depmap_glioma_recurrence_scores.set_index("CELL_LINE_NAME")

depmap_glioma_recurrence_scores["DEPMAP_RECURRENCE_SCORE"] = depmap_glioma_recurrence_scores["RECURRENCE_SCORE"]

col_order = [
    "ModelID",
    "N_UP_GENES",
    "N_DOWN_GENES",
    "UP_SCORE",
    "DOWN_SCORE",
    "DEPMAP_RECURRENCE_SCORE",
]

depmap_glioma_recurrence_scores = (depmap_glioma_recurrence_scores[col_order])

In [ ]:
depmap_glioma_recurrence_scores

In [ ]:
depmap_glioma_recurrence_scores.to_excel(
    "../results/DepMap/depmap_glioma_recurrence_scores.xlsx",
    index = True
)

- `depmap_glioma_recurrence_scores.xlsx` exported to `/results/DepMap/` directory.

## 6. Verifying variance of DepMap Recurrence Score

In [ ]:
score = depmap_glioma_recurrence_scores["DEPMAP_RECURRENCE_SCORE"]

depmap_recurrence_score_stats = pd.DataFrame()

depmap_recurrence_score_stats["N"] = [score.notna().sum()]
depmap_recurrence_score_stats["MEAN_SCORE"] = [score.mean()]
depmap_recurrence_score_stats["SD_SCORE"] = [score.std()]
depmap_recurrence_score_stats["MEDIAN_SCORE"] = [score.median()]
depmap_recurrence_score_stats["IQR"] = [score.quantile(0.75) - score.quantile(0.25)]
depmap_recurrence_score_stats["MIN"] = [score.min()]
depmap_recurrence_score_stats["MAX"] = [score.max()]
depmap_recurrence_score_stats = depmap_recurrence_score_stats.T
depmap_recurrence_score_stats.columns = [""]

In [ ]:
depmap_recurrence_score_stats

In [ ]:
fig, (ax_hist, ax_swarm) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(8, 5.5),
    sharex=True,
    gridspec_kw={"height_ratios": [3.5, 1], "hspace": 0.08},
)

sns.histplot(
    score,
    bins="fd",
    kde=True,
    ax=ax_hist,
    edgecolor="white",
    alpha=0.6,
    line_kws={"linewidth": 2},
)

ax_hist.set_ylabel("Count", fontsize=10, fontweight="bold")
ax_hist.set_title(
    f"DepMap Recurrence Score Distribution (N = {len(score)})",
    fontweight="bold",
    fontsize=12,
    pad=12,
)
ax_hist.grid(True, linestyle=":", alpha=0.5, axis="y")

sns.swarmplot(
    x=score,
    ax=ax_swarm,
    size=4.5,
    alpha=0.85,
)

ax_swarm.set_xlabel("DepMap Recurrence Score", fontsize=10, fontweight="bold")

sns.despine(ax=ax_hist)
sns.despine(ax=ax_swarm, left=True)
ax_swarm.set_yticks([])

plt.savefig(
    "../figures/depmap_glioma_recurrence_score_distribution.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
sorted_score = score.sort_values()
sorted_scores = sorted_score.to_numpy()
sorted_names = sorted_score.index.to_numpy()
ranks = np.arange(1, len(sorted_scores) + 1)

fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(ranks, sorted_scores, linewidth=1.5, zorder=2)

ax.scatter(
    ranks[:-5],
    sorted_scores[:-5],
    edgecolor="white",
    s=25,
    linewidth=0.5,
    zorder=3,
)

ax.scatter(
    ranks[-5:],
    sorted_scores[-5:],
    color="#d73027",
    edgecolor="white",
    s=45,
    linewidth=0.8,
    zorder=4,
    label="Top Hits",
)

median_val = float(np.median(sorted_scores))
ax.axhline(
    median_val,
    color="#2b5c8f",
    linestyle="--",
    linewidth=1.2,
    alpha=0.8,
    label=f"Median ({median_val:.2f})",
)

sorted_scores_min = float(sorted_scores.min())
sorted_scores_max = float(sorted_scores.max())

score_range = sorted_scores_max - sorted_scores_min
ax.set_ylim(
    sorted_scores_min - (score_range * 0.05),
    sorted_scores_max + (score_range * 0.25),
)

n_labels = 5
top_indices = np.arange(len(sorted_scores) - n_labels, len(sorted_scores))

label_offsets = [
    (-50, 15),
    (-45, 25),
    (-40, 35),
    (-35, 45),
    (-30, 55),
]

for i, (dx, dy) in zip(top_indices, label_offsets):
    ax.annotate(
        sorted_names[i],
        xy=(ranks[i], sorted_scores[i]),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=8.5,
        fontweight="bold",
        color="#222222",
        ha="right",
        va="bottom",
        arrowprops=dict(
            arrowstyle="->",
            color="#666666",
            linewidth=0.8,
            connectionstyle="arc3,rad=-0.1",
        ),
    )

ax.set_title(
    f"Ranked DepMap Glioma Recurrence Scores (N = {len(sorted_scores)})",
    fontweight="bold",
    fontsize=12,
    pad=15,
)

ax.set_xlabel("Rank (Smallest to Largest)", fontsize=10, fontweight="bold")
ax.set_ylabel("DepMap Recurrence Score", fontsize=10, fontweight="bold")

ax.grid(True, linestyle=":", alpha=0.6, axis="y")
ax.set_xlim(0, len(sorted_scores) + 2)

ax.legend(loc="lower right", frameon=True, facecolor="white", edgecolor="none")

sns.despine(ax=ax)
plt.tight_layout()

plt.savefig(
    "../figures/gdsc_glioma_recurrence_score_ranked.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 7. Preparing to create GDSC microarray Recurrence Score
- gdsc_depmap_cell_line_mapping.xlsx
    - CELL_LINE_NAME
    - GDSC_ID
    - DEPMAP_MODEL_ID
    - GDSC_EXPRESSION_AVAILABLE
    - DEPMAP_RNASEQ_AVAILABLE
- checking signature genes in GDSC expression

In [338]:
gdsc_depmap_glioma_exp = gdsc_glioma_exp.loc[gdsc_glioma_exp.index.isin(depmap_glioma_exp.index)].copy()

In [342]:
glioma_cell_line_master.head()

,STRIPPED_CELL_LINE_NAME,DEPMAP_ID,GDSC_COSMIC_ID,SANGER_MODEL_ID,CCLE_NAME,CTRPV2_NAME,DEPMAP_MODEL_ID_ALIAS,OncotreeLineage,OncotreePrimaryDisease,GDSC1_AVAILABLE,GDSC2_AVAILABLE,CTD2_AVAILABLE,CTRPV2_AVAILABLE,PRISM_AVAILABLE,GDSC_EXPRESSION_AVAILABLE,DEPMAP_EXPRESSION_AVAILABLE,NOTES
CELL_LINE_NAME,,,,,,,,,,,,,,,,,
GOS-3,GOS3,ACH-000027,NaN,NaN,GOS3_CENTRAL_NERVOUS_SYSTEM,GOS3,NaN,CNS/Brain,Adult-Type Diffuse Glioma,False,False,True,True,True,False,True,
U343,U343,ACH-000036,NaN,NaN,U343_CENTRAL_NERVOUS_SYSTEM,NaN,NaN,CNS/Brain,Adult-Type Diffuse Glioma,False,False,False,False,True,False,True,
U-118 MG,U118MG,ACH-000040,687588.0,SIDM01193,U118MG_CENTRAL_NERVOUS_SYSTEM,U118MG,NaN,CNS/Brain,Adult-Type Diffuse Glioma,True,True,True,True,True,True,True,
Hs 683,HS683,ACH-000067,1240150.0,SIDM00666,HS683_CENTRAL_NERVOUS_SYSTEM,HS683,NaN,CNS/Brain,Adult-Type Diffuse Glioma,True,True,True,True,True,True,True,
U-87 MG,U87MG,ACH-000075,687590.0,SIDM01189,U87MG_CENTRAL_NERVOUS_SYSTEM,U87MG,NaN,CNS/Brain,Adult-Type Diffuse Glioma,True,True,True,True,True,True,True,


In [345]:
gdsc_depmap_cell_line_mapping = glioma_cell_line_master[glioma_cell_line_master["DEPMAP_ID"].isin(gdsc_glioma_exp.index)].copy()

In [347]:
len(gdsc_depmap_cell_line_mapping)

52

In [351]:
print("Signature genes available in GDSC:", int(gdsc_glioma_exp.columns.isin(consensus_genes).sum()))

Signature genes available in GDSC: 250


## 8. Cross-platform validation of GDSC microarray and DepMap RNA-seq

In [352]:
GDSC_genes = gdsc_glioma_exp.columns

In [359]:
glass_gdsc_depmap_scoring_gene_universe = pd.Index(GLASS_genes.intersection(DepMap_genes).intersection(GDSC_genes))

In [360]:
glass_gdsc_depmap_common_gene_universe = pd.DataFrame(
    {
        "Metric": [
            "GLASS filtered genes",
            "GDSC genes",
            "Common genes",
            "Signature UP genes",
            "Signature DOWN genes available"
        ],
        "Value": [
            GLASS_genes.nunique(),
            DepMap_genes.nunique(),
            glass_gdsc_depmap_scoring_gene_universe.nunique(),
            glass_gdsc_depmap_scoring_gene_universe.intersection(consensus_UP_genes).nunique(),
            glass_gdsc_depmap_scoring_gene_universe.intersection(consensus_DOWN_genes).nunique()
        ],
    }
)

In [361]:
glass_gdsc_depmap_common_gene_universe

,Metric,Value
0,GLASS filtered genes,17789
1,GDSC genes,60649
2,Common genes,13004
3,Signature UP genes,152
4,Signature DOWN genes available,90


In [362]:
glass_gdsc_depmap_common_gene_universe.to_csv(
    "../results/glass_gdsc_depmap_common_gene_universe.txt",
    index = False
)

In [374]:
gdsc_depmap_common_signature_genes = pd.Index(glass_gdsc_depmap_scoring_gene_universe.intersection(consensus_genes))

In [388]:
depmap_glioma_exp_common = depmap_glioma_exp.loc[:, depmap_glioma_exp.columns.isin(gdsc_depmap_common_signature_genes)
].copy()

In [391]:
gdsc_glioma_exp_common = gdsc_glioma_exp.loc[:, gdsc_glioma_exp.columns.isin(gdsc_depmap_common_signature_genes)
].copy()

In [392]:
UP_GDSC_DepMap = pd.Index(glass_gdsc_depmap_scoring_gene_universe.intersection(consensus_UP_genes))
DOWN_GDSC_DepMap = pd.Index(glass_gdsc_depmap_scoring_gene_universe.intersection(consensus_DOWN_genes))

In [411]:
glass_crossplatform_recurrence_score_gdsc = calculate_recurrence_score(gdsc_glioma_exp_common.T, UP_GDSC_DepMap, DOWN_GDSC_DepMap)
glass_crossplatform_recurrence_score_gdsc = glass_crossplatform_recurrence_score_gdsc.drop(columns=["UP_SCORE", "DOWN_SCORE"])
glass_crossplatform_recurrence_score_gdsc = glass_crossplatform_recurrence_score_gdsc.rename(columns={"RECURRENCE_SCORE": "GDSC_RECURRENCE_SCORE_CROSSPLATFORM"})

glass_crossplatform_recurrence_score_depmap = calculate_recurrence_score(depmap_glioma_exp_common.T, UP_GDSC_DepMap, DOWN_GDSC_DepMap)
glass_crossplatform_recurrence_score_depmap = glass_crossplatform_recurrence_score_depmap.drop(columns=["UP_SCORE", "DOWN_SCORE"])
glass_crossplatform_recurrence_score_depmap = glass_crossplatform_recurrence_score_depmap.rename(columns={"RECURRENCE_SCORE": "DEPMAP_RECURRENCE_SCORE_CROSSPLATFORM"})

In [412]:
glass_crossplatform_recurrence_score_gdsc.head()

,GDSC_RECURRENCE_SCORE_CROSSPLATFORM
ModelID,
ACH-000883,-0.169298
ACH-000437,-0.196491
ACH-001016,-0.087281
ACH-002257,-0.241082
ACH-000098,-0.097661


In [413]:
depmap_gdsc_glioma_recurrence_scores = glass_crossplatform_recurrence_score_gdsc.merge(
    glass_crossplatform_recurrence_score_depmap,
    on="ModelID",
    how="inner"
)

In [414]:
gdsc_exp_common = (
    gdsc_glioma_exp_common.loc[
        gdsc_glioma_exp_common.index.isin(
            depmap_gdsc_glioma_recurrence_scores.index
        )
    ].copy()
)

depmap_exp_common = (
    depmap_glioma_exp_common.loc[
        depmap_glioma_exp_common.index.isin(
            depmap_gdsc_glioma_recurrence_scores.index
        )
    ].copy()
)

In [415]:
up_genes_present = (
    pd.Index(UP_GDSC_DepMap)
    .intersection(depmap_exp_common.columns)
    .intersection(gdsc_exp_common.columns)
)

depmap_up_present = depmap_exp_common.loc[:, up_genes_present].notna()
gdsc_up_present = gdsc_exp_common.loc[:, up_genes_present].notna()

n_up_genes = (
    (depmap_up_present & gdsc_up_present)
    .sum(axis=1)
)

n_up_genes.index.name = "DEPMAP_ID"

depmap_gdsc_glioma_recurrence_scores["N_UP_GENES"] = (
    n_up_genes.reindex(
        depmap_gdsc_glioma_recurrence_scores.index
    )
)

down_genes_present = (
    pd.Index(DOWN_GDSC_DepMap)
    .intersection(depmap_exp_common.columns)
    .intersection(gdsc_exp_common.columns)
)

depmap_down_present = depmap_exp_common.loc[:, down_genes_present].notna()
gdsc_down_present = gdsc_exp_common.loc[:, down_genes_present].notna()

n_down_genes = (
    (depmap_down_present & gdsc_down_present)
    .sum(axis=1)
)

n_down_genes.index.name = "DEPMAP_ID"

depmap_gdsc_glioma_recurrence_scores["N_DOWN_GENES"] = (
    n_down_genes.reindex(
        depmap_gdsc_glioma_recurrence_scores.index
    )
)

In [416]:
depmap_gdsc_glioma_recurrence_scores.head()

,GDSC_RECURRENCE_SCORE_CROSSPLATFORM,DEPMAP_RECURRENCE_SCORE_CROSSPLATFORM,N_UP_GENES,N_DOWN_GENES
ModelID,,,,
ACH-000883,-0.169298,-0.196491,152,90
ACH-000437,-0.196491,-0.157383,152,90
ACH-001016,-0.087281,-0.157675,152,90
ACH-000098,-0.097661,-0.160526,152,90
ACH-000102,-0.193640,-0.195906,152,90
